# Setup

In [ ]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [ ]:
isin = pd.read_excel('Data\\EA_ISINs.xlsx')

In [ ]:
unique_isin = tuple(isin['ISIN'])

In [ ]:
treasury = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [ ]:
unique_treasury = tuple(treasury['ISIN'].unique())

In [ ]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [ ]:
hf_overlap = tuple(hedge_funds['entity_id'].unique())

# SFTDS fund bond day panel, sftds_dataframe.csv and sftds_hedgefunds.csv

## final - including whatever identified in overlap analysis

In [ ]:
# Data prep
query = f"""

WITH ky_bm_borrowers AS (
    SELECT DISTINCT borrower_id
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
    WHERE borrower_country_residence IN ('KY', 'BM')
)
SELECT 
    s.business_date,
    s.borrower_id AS entity_id,
    MAX(CASE WHEN s_borrower.sector = 'BANK' THEN 1 ELSE 0 END) AS bank_indicator,
    s.security_isin,
    SUM(s.nominal_value) AS borrowing_volume,
    AVG(repo_rate) as borrowing_rate, AVG(CASE WHEN contractual_maturity < 1 THEN 1 ELSE contractual_maturity END) as borrowing_term
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.nominal_ccy IN ('EUR', 'USD')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND (
        s.borrower_id IN (SELECT borrower_id FROM ky_bm_borrowers)
        OR s_borrower.if_strategy = 'Hedge'
        OR borrower_id IN {hf_overlap}
        OR (s_borrower.sector = 'BANK' AND borrower_country = 'GB') 
    )
    AND (
        (s.security_isin IN {unique_isin} AND s.nominal_ccy = 'EUR')
        OR (s.security_isin IN {unique_treasury} AND s.nominal_ccy = 'USD')
    )
GROUP BY s.business_date, s.borrower_id, s.security_isin
ORDER BY s.business_date, s.borrower_id, s.security_isin

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

WITH ky_bm_lenders AS (
    SELECT DISTINCT lender_id
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
    WHERE lender_country_residence IN ('KY', 'BM')
)
SELECT 
    s.business_date,
    s.lender_id AS entity_id,
    MAX(CASE WHEN s_lender.sector = 'BANK' THEN 1 ELSE 0 END) AS bank_indicator,
    s.security_isin,
    SUM(s.nominal_value) AS lending_volume,
    AVG(repo_rate) as lending_rate, AVG(CASE WHEN contractual_maturity < 1 THEN 1 ELSE contractual_maturity END) as lending_term
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.nominal_ccy IN ('EUR', 'USD')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND (
        s.lender_id IN (SELECT lender_id FROM ky_bm_lenders)
        OR s_lender.if_strategy = 'Hedge'
        OR s.lender_id IN {hf_overlap}
        OR (s_lender.sector = 'BANK' AND lender_country = 'GB')
    )
    AND (
        (s.security_isin IN {unique_isin} AND s.nominal_ccy = 'EUR')
        OR (s.security_isin IN {unique_treasury} AND s.nominal_ccy = 'USD')
    )
GROUP BY s.business_date, s.lender_id, s.security_isin
ORDER BY s.business_date, s.lender_id, s.security_isin
"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df = df_borrowing.merge(df_lending, on= ['business_date', 'entity_id', 'bank_indicator', 'security_isin'], how = 'outer')

In [ ]:
len(df['entity_id'].unique())

In [ ]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [ ]:
df['borrowing_volume'].fillna(0,inplace=True)
df['lending_volume'].fillna(0,inplace=True)

In [ ]:
pd.DataFrame(df['entity_id'].unique()).to_csv('key dataframe\\sftds_hedgefunds.csv')

In [ ]:
df.to_csv('key dataframe\\sftds_dataframe.csv')

# EMIR fund contract day panel, emir_dataframe.csv and german_futures.csv

In [ ]:
# Data prep
query = f"""

SELECT reference_period as business_date, notional_leg1_currency as currency, other_cpty_id as entity_id, MAX(CASE WHEN s_entity.sector = 'BANK' THEN 1 ELSE 0 END) AS bank_indicator,
 product_isin, underlying_id, underlying_maturity_date, underlying_index_name, maturity_date, CASE WHEN direction = 'MAKE' THEN 'SLLR' WHEN direction = 'TAKE' THEN 'BYER' ELSE direction END AS direction, SUM(notional_leg1) as vol
FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
WHERE reference_period >= '2024-04-29'
AND reference_period <= '2025-12-31'
AND notional_leg1 < 1e18
AND asset_class = 'INTR'
AND contract_type = 'FUTR'
AND notional_leg1 > 0
AND notional_leg1_currency IN ("EUR", "USD")
AND LEFT(product_cfi, 3) = 'FFD'
AND (LEFT(underlying_id, 2) != 'EU' OR underlying_id IS NULL)
AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
AND MONTH(maturity_date) IN (3, 6, 9, 12)
AND (s_entity.sector = 'IF' OR other_cpty_id_esa_sector_riad = 'S124' OR other_cpty_id IN {hf_overlap} OR (s_entity.sector = 'BANK' AND other_cpty_id_country_code_gleif = 'GB'))
GROUP BY reference_period, currency, other_cpty_id, product_isin, underlying_id, underlying_maturity_date, underlying_index_name, maturity_date, direction
ORDER BY reference_period, currency, other_cpty_id, product_isin, underlying_id, underlying_maturity_date, underlying_index_name, maturity_date, direction

"""

df_emir_refit = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""


SELECT reference_period as business_date, notional_currency1 as currency, other_cpty_id as entity_id, MAX(CASE WHEN s_entity.sector = 'BANK' THEN 1 ELSE 0 END) AS bank_indicator,
 product_id as product_isin, underlying_id, underlying_maturity_date, underlying_index_name, maturity_date, counterparty_side as direction, SUM(notional) as vol
FROM crp_emir_ecb.emir_ecb_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
WHERE reference_period >= '2021-01-04'
AND reference_period <= '2024-04-29'
AND notional < 1e18
AND asset_class = 'INTR'
AND contract_type = 'FUTR'
AND notional > 0
AND notional_currency1 IN ("EUR", "USD")
AND LEFT(product_clssfctn, 3) = 'FFD'
AND ((LEFT(underlying_id, 2) NOT IN ('EU', 'XC')) OR underlying_id IS NULL)
AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
AND MONTH(maturity_date) IN (3, 6, 9, 12)
AND (s_entity.sector = 'IF' OR s_entity.sector = 'OFI' OR s_entity.sector = 'OTHR' OR other_cpty_id_esa_sector = 'S124' OR other_cpty_id IN {hf_overlap} OR (s_entity.sector = 'BANK' AND other_cpty_id_country_code = 'GB'))
GROUP BY reference_period, currency, other_cpty_id, product_isin, underlying_id, underlying_maturity_date, underlying_index_name, maturity_date, direction
ORDER BY reference_period, currency, other_cpty_id, product_isin, underlying_id, underlying_maturity_date, underlying_index_name, maturity_date, direction


"""

df_emir_old = pd.read_sql_query(query, cnxn)

In [ ]:
df_emir = pd.concat([df_emir_refit, df_emir_old]).drop_duplicates().reset_index(drop=True)

## Underlying index Name

In [ ]:
# Mapping from underlying_index_name to short futures name
name_to_future = {}

# --- EUR Bond Futures ---
schatz_names = [
    'EURX ESCHATZ', 'EURO SCHATZ', 'Euro Schatz', 'Euro Schatz-Future',
    'Euro-Schatz', 'EURO - SCHATZ', 'Euro Schatz -2 Year Bond',
    'Euro-Schatz-2-Year-Bond'
]
for n in schatz_names:
    name_to_future[n] = 'schatz'

bobl_names = [
    'EURX EURBOBL', 'EURO BOBL', 'Euro Bobl -5 Year Bond', 'Euro-Bobl',
    'Euro-Bobl Futures (FGBM)', 'Euro-Bobl Futures',
    'Euro-Bobl-5-Year-Bond', 'Euro-Bobl 5yr 6%'
]
for n in bobl_names:
    name_to_future[n] = 'bobl'

bund_names = [
    'EURO BUND', '"EURX EUR-BUND"', 'Euro Bund -10 Year Bond', 'Euro-Bund',
    'Euro-Bund Futures', 'Euro bund', 'Euro Bund', 'BUND', 'Euro Bund-Future',
    'FGBL', 'EURO BUND FUTURES', 'EURO EU BOND EUROPEAN LT UND',
    'BUND FUTURES SEP24', 'BUND FUTURES DEC24', 'BUND FUTURES MAR25',
    'BUND FUTURES JUN25', 'FGBL MAR25 [FIXED INCOME FUTURES',
    'FGBL SI 20250606 PS', 'FGBL SI 20250908 PS', 'FGBL SI 20251208 PS',
    'Euro-Bund-10-Year-Bond', 'Euro-Bund 10yr 6%'
]
for n in bund_names:
    name_to_future[n] = 'bund'

buxl_names = [
    '"EURX EURO-BUXL"', 'GERMAN GOVERNMENT BUXL  Bond',
    'Euro Buxl -15 - 30 Year Bond', 'Euro-Buxl Futures',
    'Euro Buxl Futures', 'Euro Buxl', 'Euro-Buxl',
    'Euro-Buxl-15-30-Year-Bond'
]
for n in buxl_names:
    name_to_future[n] = 'buxl'

btp_names = [
    'EURO-BTP LONGTERM ITA GOV BOND', 'Euro-BTP Italian Government Bond',
    'Euro BTP Futures', '"EUX EURO-BTP"', 'Long-Term Euro-BTP',
    'BTP FUTURES JUN24', 'BTP FUTURES SEP24', 'BTP FUTURES DEC24',
    'BTP FUTURES MAR25', 'BTP FUTURES JUN25', 'BTP FUTURES SEP25',
    'Euro-BTP-Italian-Government-Bond', 'Euro BTP Futures'
]
for n in btp_names:
    name_to_future[n] = 'btp'

short_btp_names = [
    'SHORT TERM EURO-BTP  bond', 'Short-Term Euro-BTP Italian Government Bond',
    'SHORT TERM BTP FUTURES', 'Short-Term Euro-BTP',
    'Short-Term-Euro-BTP-Italian-Government-Bond'
]
for n in short_btp_names:
    name_to_future[n] = 'short_btp'

oat_names = [
    'EURO-OAT bond', 'Euro-OAT French Government Bond', 'FOAT', 'EUX FOAT',
    'EURO-OAT FUTURES', 'Euro-OAT', 'Euro-OAT Futures',
    'Euro-OAT-French-Government-Bond'
]
for n in oat_names:
    name_to_future[n] = 'oat'

bono_names = [
    'EURO-BONO SPANISH GOV BOND',
    'Euro-BONO-Futures-on-Long-term-Spanish'
]
for n in bono_names:
    name_to_future[n] = 'bono'

# --- USD Bond Futures ---
tn2_names = [
    '"CBT 2Y T-NOTE"', '2YUSTN bond', '2 Year US Treasury Note',
    'CBT 2 Year Treasury Notes', 'CBOT 2-YEAR TREAS. NOTE', '2-Year T-Note',
    'CBOT-TWO-YEAR-TREASURY-NOTE-FUTURE', '2-Year T-Not',
    '2-Year-US-Treasury-Note', '2-Year-Micro-Treasury-Yield'
]
for n in tn2_names:
    name_to_future[n] = 'tn2'

tn3_names = [
    'THREE YEARS US T-NOTE',
    '3-YEAR-US-TREASURY-NOTE'
]
for n in tn3_names:
    name_to_future[n] = 'tn3'

tn5_names = [
    '"CBT 5Y T-NOTE"', 'Five years US T-Note bond', '5 Year US Treasury Note',
    'CBT 5 Year Treasury Notes', 'CBOT 5 YEAR TREAS. NOTE',
    'USTREASNOTE5YR', '5-Year T-Note', '5-Year-US-Treasury-Note',
    'CBT 5Y T-NO', '5-Year-Micro-Treasury-Yield'
]
for n in tn5_names:
    name_to_future[n] = 'tn5'

tn10_names = [
    '"CBT 10Y T-NOTE"', 'Ten years US T-Note bond', '10 Year US Treasury Note',
    'CBT 10 Year Treasury Notes', '10yr TREASURY NOTES',
    'USTREASNOTE10YR', '10-Year T-Note', '10-Year-US-Treasury-Note',
    'CBT 10Y T-NO', 'CBOT 10Y T-NOTE', '10-Year-Micro-Treasury-Yield'
]
for n in tn10_names:
    name_to_future[n] = 'tn10'

tbond_names = [
    '"CBT T-BONDS"', 'US Treasury Bond bond', 'US Treasury Bond',
    'C.B.O.T. TREASURY BONDS', 'U.S. Treasury Bond',
    '30-Year-US-Treasury-Bond', 'CBOT T-BONDS',
    'CBOT 30 YR TRSY BOND', '30-Year-Micro-Treasury-Yield'
]
for n in tbond_names:
    name_to_future[n] = 'tbond'

ultra10_names = [
    'CBT ULT TNOTE', 'ULTRA 10YR US TREASURY NOTE St',
    'Ultra 10-Year US Treasury Note', 'ULTRA 10YR TNOTE',
    'CBT Ultra 10-Year U.S. Treasury', 'ULTRA 10Y US T-NOTE',
    'Ultra 10 year US Treasury', 'Ultra 10-Year U.S. Treasu',
    'ULTRA-10-YR-T-NOTE-FUTURE', 'Ultra-10-Year-US-Treasury-Note',
    'USUltraTreasuryNote10Years', 'CBT ULT TNO',
    'US Ultra Treasury Note 10 Years', 'Micro E',
    'CBOT ULTRA 10YR TRSY'
]
for n in ultra10_names:
    name_to_future[n] = 'ultra10'

ultrabond_names = [
    'CBT UL TBONDS', 'ULTRA US TREASURY BOND STOCK',
    'Ultra Treasury Bond', 'CBT ULTRA US TREASURY BOND FUTURES',
    'LONG-TERM ULTRA U.S. T bond', 'ULTRA T-BOND',
    'UL-TERM US TREASURY BOND', 'Ultra U.S. Treasury Bond',
    'ULTRA-U.S.-TREASURY-BOND-FUTURES', 'Ultra-Treasury-Bond',
    'ULTRA TRSY BOND'
]
for n in ultrabond_names:
    name_to_future[n] = 'ultrabond'

# --- Rate Futures (flagged for removal) ---
rate_futures_names = [
    'THREE-MONTH SOFR FUTURES', 'THREEMONTHSOFRFUTURES',
    'Three-Month SOFR Stock', 'SOFR', 'THREE-MONTH SOFR', 'SR3',
    'THREEMONTH SOFR FUTURES', 'EU0009652783-SOFR',
    'Secured Overnight Financing Rate 3-month average o',
    'Secured-Overnight-Financing-Rate-3-month-average',
    'UNITED STATES SOFR SECURED OVERNIGHT FINANCING RATE',
    'United States SOFR Secure', 'CME THREE', 'CME THREE-MONTH SOFR FUTURES',
    '30-DAY INT RATE(FEDFUND', '30-Day-Fed-Funds', 'CBT 30D INT',
    'FEDERAL FUNDS EFFECTIVE R',
    'LIFFE EURIBOR 3 MONTH', 'EURIBOR 3MO', 'EU0009652783-EURI',
    'EURODOLLAR', '90 DAY EURODOLLAR', 'GLOBEX-Euro-Dollar',
    'Libor USD Fix 3 month', 'LIBOR USD FIX 3 MONTH',
    'ESTR Volume Weighted Trimmed Mean Rate',
    'EU0009652783'
]
for n in rate_futures_names:
    name_to_future[n] = 'RATE_FUTURE_REMOVE'

In [ ]:
# --- Apply mapping ---
df_emir['future_short'] = df_emir['underlying_index_name'].map(name_to_future)

In [ ]:
# Build futures_contract from maturity_date
df_emir['maturity_month'] = pd.to_datetime(df_emir['maturity_date']).dt.strftime('%m_%y')
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
df_emir['vol'][df_emir['future_short'] == 'RATE_FUTURE_REMOVE'].sum()/df_emir['vol'].sum()

In [ ]:
#df_emir = df_emir[df_emir['future_short'] != 'RATE_FUTURE_REMOVE']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

## Constant Futures ISINs from Eurex

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
# Mapping from underlying_id (ISIN) to short futures name
isin_to_future = {
    'DE0009652669': 'schatz',
    'DE0009652651': 'bobl',
    'DE0009652644': 'bund',
    'DE0009652636': 'buxl',
    'DE000A0ZW3V8': 'btp',
    'DE000A1EZJ09': 'short_btp',
    'DE000A1MAPW3': 'oat',
    'DE000A163W29': 'bono',
}

# Fill in future_short from underlying_id where it's still missing or marked for fallback
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'product_isin'].map(isin_to_future)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
# Mapping from underlying_id (ISIN) to short futures name
isin_to_future = {
    'DE0009652669': 'schatz',
    'DE0009652651': 'bobl',
    'DE0009652644': 'bund',
    'DE0009652636': 'buxl',
    'DE000A0ZW3V8': 'btp',
    'DE000A1EZJ09': 'short_btp',
    'DE000A1MAPW3': 'oat',
    'DE000A163W29': 'bono',
}

# Fill in future_short from underlying_id where it's still missing or marked for fallback
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'underlying_id'].map(isin_to_future)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

In [ ]:
#df_emir = df_emir[df_emir['underlying_id'] != 'UKWN']

In [ ]:
#df_emir = df_emir[(df_emir['underlying_id'].isna() == False) | (df_emir['underlying_index_name'].isna() == False)]

## Futures ISINs newest contracts (from Davide)

In [ ]:
# Futures contract ISINs -> future_short
futures_isin_to_future = {
    # Bund (RX)
    'DE000F2MGDC1': 'bund', 'DE000F2Y2EY1': 'bund', 'DE000F3A9ET3': 'bund',
    # Bobl (OE)
    'DE000F1B2NH5': 'bobl', 'DE000F1NGF61': 'bobl', 'DE000F1ZLHQ6': 'bobl',
    'DE000F2MGDD9': 'bobl', 'DE000F2Y2EZ8': 'bobl', 'DE000F3A9EU1': 'bobl',
    # Schatz (DU)
    'DE000F1B2NJ1': 'schatz', 'DE000F1NGF79': 'schatz', 'DE000F1ZLHR4': 'schatz',
    'DE000F2MGDE7': 'schatz', 'DE000F2Y2E04': 'schatz', 'DE000F3A9EV9': 'schatz',
    # BTP (IK)
    'DE000F1B2NF9': 'btp', 'DE000F1NGF38': 'btp', 'DE000F1ZLHN3': 'btp',
    'DE000F2MGDA5': 'btp', 'DE000F2Y2EW5': 'btp', 'DE000F3A9ER7': 'btp',
    # OAT
    'DE000F1B2NL7': 'oat', 'DE000F1NGGA8': 'oat', 'DE000F1ZLHU8': 'oat',
    'DE000F2MGDH0': 'oat', 'DE000F2Y2E38': 'oat', 'DE000F3A9EY3': 'oat',
    # Buxl (UB)
    'DE000F1B2NK9': 'buxl', 'DE000F1NGF87': 'buxl', 'DE000F1ZLHS2': 'buxl',
    'DE000F2MGDF4': 'buxl', 'DE000F2Y2E12': 'buxl', 'DE000F3A9EW7': 'buxl',
    # Short BTP (BTS)
    'DE000F1B2NE2': 'short_btp', 'DE000F1NGF46': 'short_btp', 'DE000F1ZLHM5': 'short_btp',
    'DE000F2MGDB3': 'short_btp', 'DE000F2Y2EX3': 'short_btp', 'DE000F3A9ES5': 'short_btp',
}

# Apply where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'product_isin'].map(futures_isin_to_future)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
# Futures contract ISINs -> future_short
futures_isin_to_future = {
    # Bund (RX)
    'DE000F2MGDC1': 'bund', 'DE000F2Y2EY1': 'bund', 'DE000F3A9ET3': 'bund',
    # Bobl (OE)
    'DE000F1B2NH5': 'bobl', 'DE000F1NGF61': 'bobl', 'DE000F1ZLHQ6': 'bobl',
    'DE000F2MGDD9': 'bobl', 'DE000F2Y2EZ8': 'bobl', 'DE000F3A9EU1': 'bobl',
    # Schatz (DU)
    'DE000F1B2NJ1': 'schatz', 'DE000F1NGF79': 'schatz', 'DE000F1ZLHR4': 'schatz',
    'DE000F2MGDE7': 'schatz', 'DE000F2Y2E04': 'schatz', 'DE000F3A9EV9': 'schatz',
    # BTP (IK)
    'DE000F1B2NF9': 'btp', 'DE000F1NGF38': 'btp', 'DE000F1ZLHN3': 'btp',
    'DE000F2MGDA5': 'btp', 'DE000F2Y2EW5': 'btp', 'DE000F3A9ER7': 'btp',
    # OAT
    'DE000F1B2NL7': 'oat', 'DE000F1NGGA8': 'oat', 'DE000F1ZLHU8': 'oat',
    'DE000F2MGDH0': 'oat', 'DE000F2Y2E38': 'oat', 'DE000F3A9EY3': 'oat',
    # Buxl (UB)
    'DE000F1B2NK9': 'buxl', 'DE000F1NGF87': 'buxl', 'DE000F1ZLHS2': 'buxl',
    'DE000F2MGDF4': 'buxl', 'DE000F2Y2E12': 'buxl', 'DE000F3A9EW7': 'buxl',
    # Short BTP (BTS)
    'DE000F1B2NE2': 'short_btp', 'DE000F1NGF46': 'short_btp', 'DE000F1ZLHM5': 'short_btp',
    'DE000F2MGDB3': 'short_btp', 'DE000F2Y2EX3': 'short_btp', 'DE000F3A9ES5': 'short_btp',
}

# Apply where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'underlying_id'].map(futures_isin_to_future)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

## doubling down on product ISIN

In [ ]:
isin_map = {
    # BTP
    'DE000C75XMS2': 'btp', 'DE000F0FSQL7': 'btp', 'DE000F01NAB3': 'btp',
    'DE000C7F2X85': 'btp', 'DE000C7PB9W9': 'btp', 'DE000C7X7UH4': 'btp',
    'DE000C7X7UJ0': 'btp',
    # Short BTP
    'DE000C75XMT0': 'short_btp', 'DE000F0FSQM5': 'short_btp', 'DE000F0Q7BT8': 'short_btp',
    'DE000F01NAC1': 'short_btp', 'DE000C5GTXA4': 'short_btp', 'DE000C5RQD94': 'short_btp',
    'DE000C52GUD1': 'short_btp', 'DE000C1T6JC2': 'short_btp', 'DE000C6J0K03': 'short_btp',
    'DE000C6R0437': 'short_btp', 'DE000C6YTCL6': 'short_btp', 'DE000C671E14': 'short_btp',
    'DE000C7F2X93': 'short_btp', 'DE000C7PB9X7': 'short_btp',
    # Bobl
    'DE000C75XMV6': 'bobl', 'DE000C5GTXC0': 'bobl', 'DE000C5RQDB7': 'bobl',
    'DE000C6R0460': 'bobl', 'DE000C6YTCN2': 'bobl', 'DE000C671E30': 'bobl',
    'DE000C7F2YB2': 'bobl', 'DE000C7PB9Z2': 'bobl', 'DE000C7X7UL6': 'bobl',
    # Buxl
    'DE000C75XMX2': 'buxl', 'DE000F0FSQR4': 'buxl', 'DE000F01NAG2': 'buxl',
    'DE000C5GTXE6': 'buxl', 'DE000C5RQDD3': 'buxl', 'DE000C6R0494': 'buxl',
    'DE000C6YTCQ5': 'buxl', 'DE000C671E55': 'buxl', 'DE000C7F2YD8': 'buxl',
    'DE000C7PB918': 'buxl', 'DE000C7X7UN2': 'buxl',
    # Schatz
    'DE000BU22049': 'schatz', 'DE000C5GTXD8': 'schatz', 'DE000C5RQDC5': 'schatz',
    'DE000C6R0478': 'schatz', 'DE000C6YTCP7': 'schatz', 'DE000C671E48': 'schatz',
    'DE000C7F2YC0': 'schatz', 'DE000C7PB900': 'schatz', 'DE000C7X7UM4': 'schatz',
    'DE000F01NAE7': 'schatz', 'DE000F01NAF4': 'schatz',
    # Bund
    'DE000BU2Z007': 'bund', 'DE000C5RQDA9': 'bund', 'DE000C671E22': 'bund',
    'DE000C7F2YA4': 'bund', 'DE000C7PB9Y5': 'bund', 'DE000C7X7UK8': 'bund',
    'DE000F1NGF20': 'bund', 'DE0009664581': 'bund',
    # OAT
    'DE000F0FSQT0': 'oat', 'DE000F01NAJ6': 'oat', 'DE000C5GTXG1': 'oat',
    'DE000C5RQDF8': 'oat', 'DE000C6R05A8': 'oat', 'DE000C6YTCS1': 'oat',
    'DE000C671E71': 'oat', 'DE000C7F2YF3': 'oat', 'DE000C7PB934': 'oat',
    'DE000C7X7UP7': 'oat',
    # Bono
    'DE000F0FSQJ1': 'bono', 'DE000F01M991': 'bono', 'DE000F1ZLHL7': 'bono',
    'DE000F2MGC80': 'bono',
    # US Treasuries
    'US12573H3286': 'tn10', 'XC0009690196': 'tn10',
    'US12573H2866': 'tn2',  'XC0009690675': 'tn2', 'XC0001717096' :'tn2',
    'US12573L7174': 'tn5',
    'US12573H2601': 'tbond', 'US12573H3104': 'tbond',
    # Rate futures -> remove
    'GB00F71K4T03': 'RATE_FUTURE_REMOVE', 'GB00F7B19G51': 'RATE_FUTURE_REMOVE',
    'GB00H3PJDL87': 'RATE_FUTURE_REMOVE', 'GB00HB9WVH19': 'RATE_FUTURE_REMOVE',
    'GB00J14H8C30': 'RATE_FUTURE_REMOVE', 'GB00J8R2ZN46': 'RATE_FUTURE_REMOVE',
    'GB00JLVCWV80': 'RATE_FUTURE_REMOVE', 'XC0009654242': 'RATE_FUTURE_REMOVE',
    'GB00H6CZ4W89': 'RATE_FUTURE_REMOVE', 'GB00H7QJ2K79': 'RATE_FUTURE_REMOVE',
    'GB00J7WDF916': 'RATE_FUTURE_REMOVE', 'GB00J8WDXS35': 'RATE_FUTURE_REMOVE',
    'GB00JBVLWR12': 'RATE_FUTURE_REMOVE', 'GB00JH6LTC25': 'RATE_FUTURE_REMOVE',
    'GB00K0RJJF84': 'RATE_FUTURE_REMOVE', 'GB00K6Z0PG84': 'RATE_FUTURE_REMOVE',
    'GB00K8XLLR03': 'RATE_FUTURE_REMOVE', 'GB00KFMTR435': 'RATE_FUTURE_REMOVE',
    'GB00KHJT1489': 'RATE_FUTURE_REMOVE', 'GB00HBCK1X80': 'RATE_FUTURE_REMOVE',
    'GB00F7B6VM05': 'RATE_FUTURE_REMOVE', 'GB00H46DQ065': 'RATE_FUTURE_REMOVE',
    'GB00H7QLC142': 'RATE_FUTURE_REMOVE', 'GB00HB9ZR833': 'RATE_FUTURE_REMOVE',
    'GB00HBD02M38': 'RATE_FUTURE_REMOVE', 'GB00J14RR372': 'RATE_FUTURE_REMOVE',
    'GB00J8WFXD48': 'RATE_FUTURE_REMOVE', 'GB00JBVNM225': 'RATE_FUTURE_REMOVE',
    'GB00JH6MVF27': 'RATE_FUTURE_REMOVE', 'GB00JLVFXD62': 'RATE_FUTURE_REMOVE',
    'GB00K0RQK019': 'RATE_FUTURE_REMOVE', 'GB00K6Z4G010': 'RATE_FUTURE_REMOVE',
    'GB00KKYGSW29': 'RATE_FUTURE_REMOVE', 'GB00KKYGSY43': 'RATE_FUTURE_REMOVE',
    'GB00JPF3HL01': 'RATE_FUTURE_REMOVE', 'GB00KHK64G15': 'RATE_FUTURE_REMOVE',
    'DE0009653147': 'RATE_FUTURE_REMOVE', 'DE000C32PBL7': 'RATE_FUTURE_REMOVE',
    'US81379LAA44': 'RATE_FUTURE_REMOVE', 'XC0009691509': 'RATE_FUTURE_REMOVE',
    'GB00H1QMMS28': 'RATE_FUTURE_REMOVE', 'GB00H1QMMT35': 'RATE_FUTURE_REMOVE',
    'GB00H1QMMV56': 'RATE_FUTURE_REMOVE', 'GB00H1QMMX70': 'RATE_FUTURE_REMOVE',
    'GB00H1QMMY87': 'RATE_FUTURE_REMOVE', 'GB00H1QMMZ94': 'RATE_FUTURE_REMOVE',
    'GB00H1QMN348': 'RATE_FUTURE_REMOVE', 'GB00H1QMND40': 'RATE_FUTURE_REMOVE',
    'GB00H1QMNJ02': 'RATE_FUTURE_REMOVE', 'GB00H1QMNK17': 'RATE_FUTURE_REMOVE',
    'GB00H1QMNL24': 'RATE_FUTURE_REMOVE', 'GB00F7595128': 'RATE_FUTURE_REMOVE',
    'GB00H221MZ78': 'RATE_FUTURE_REMOVE', 'GB00H37S3355': 'RATE_FUTURE_REMOVE',
    'GB00H46M2162': 'RATE_FUTURE_REMOVE', 'GB00H46M2279': 'RATE_FUTURE_REMOVE',
    'GB00H46M2386': 'RATE_FUTURE_REMOVE', 'GB00H46M2493': 'RATE_FUTURE_REMOVE',
    'GB00H46M2501': 'RATE_FUTURE_REMOVE', 'GB00H46M2618': 'RATE_FUTURE_REMOVE',
    'GB00H46M2725': 'RATE_FUTURE_REMOVE', 'GB00H46M2832': 'RATE_FUTURE_REMOVE',
    'GB00H46M2949': 'RATE_FUTURE_REMOVE', 'GB00H46M2B66': 'RATE_FUTURE_REMOVE',
    'GB00H7NBF416': 'RATE_FUTURE_REMOVE', 'GB00J8R2ZQ76': 'RATE_FUTURE_REMOVE',
    'GB00H46M2H29': 'RATE_FUTURE_REMOVE', 'XD0416719934': 'RATE_FUTURE_REMOVE',
    'XD0416719736': 'RATE_FUTURE_REMOVE',
    # Not bond futures
    'US90390T1051': 'not_bond_futures',  # commodity
    'DE000C52VJW3': 'not_bond_futures',  # equity
    'IR': 'not_bond_futures',            # interest rate (generic)
}

In [ ]:
# Apply where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'product_isin'].map(isin_map)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
# Apply where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'underlying_id'].map(isin_map)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

## any additional futures based on patterns in ISINs

In [ ]:
# Look at the unmapped DE, IT, US ISINs
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')

unmapped_isins = df_emir.loc[needs_fill, 'underlying_id'].unique()

# Check: are any of these in the deliverables baskets as deliverable bonds?
# If NOT, and they start with DE000F or similar patterns, they might be contract ISINs
print("Unmapped ISINs starting with DE000F (likely contract ISINs):")
print([x for x in unmapped_isins if isinstance(x, str) and x.startswith('DE000F')])

print("\nAll unmapped DE ISINs:")
print(sorted([x for x in unmapped_isins if isinstance(x, str) and x.startswith('DE')]))

print("\nAll unmapped IT ISINs:")
print(sorted([x for x in unmapped_isins if isinstance(x, str) and x.startswith('IT')]))

print("\nAll unmapped US ISINs:")
print(sorted([x for x in unmapped_isins if isinstance(x, str) and x.startswith('US')]))

In [ ]:
# googled the ones in the top line and found these
historical_contract_isins = {
    'DE000F0FSQN3': 'bund',
    'DE000F0FSQP8': 'bobl',
    'DE000F0FSQQ6': 'schatz',
    'DE000F0Q7BW2': 'schatz',
    'DE000F0Q7BV4': 'bund',
    'DE000F0Q7BU6': 'bund',
    'DE000F0Q7BS0': 'btp',
    'DE000F0Q7BZ5': 'oat',
    'DE000F0Q7BX0': 'buxl',
    'DE000F01NAD9': 'bund',
    'DE000F01NAF4': 'schatz',
    'DE000F1B2NG7': 'bund',
    'DE000F1NGF53': 'bund',
    'DE000F1ZLHP8': 'bund',
}

In [ ]:
# Apply where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'product_isin'].map(historical_contract_isins)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
# Apply where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'underlying_id'].map(historical_contract_isins)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

## Matching US deliverables

In [ ]:
deliverables = pd.read_csv('Data\\Deliverables_US.csv')

In [ ]:
cusip_translation = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [ ]:
deliverables['Contract'].str[:3].unique()

In [ ]:
cusip_translation.head()

In [ ]:
import re
import pandas as pd

us_prefix_map = {
    'TY': 'tn10', 'FV': 'tn5', 'TU': 'tn2',
    'UXY': 'ultra10', 'WN': 'ultrabond', 'US': 'tbond',
}
month_map = {'H': '03', 'M': '06', 'U': '09', 'Z': '12'}

def parse_us_contract(contract):
    match = re.match(r'^([A-Z]+)([HMUZ])(\d{1,2})$', contract)
    if not match:
        return None, None
    prefix, month_letter, year = match.group(1), match.group(2), match.group(3)
    future_short = us_prefix_map.get(prefix)
    month = month_map[month_letter]
    if len(year) == 1:
        year = '2' + year
    return future_short, f"{month}_{year}"

# Parse the Deliverable string: "CUSIP Govt FACTOR CUSIP Govt FACTOR ..."
# Capture CUSIP (8 alphanumerics) followed by " Govt " followed by a float factor.
pattern = re.compile(r'([A-Z0-9]{8})\s+Govt\s+([\d.]+)')

rows = []
for _, r in deliverables.iterrows():
    future_short, maturity_month = parse_us_contract(r['Contract'])
    if future_short is None:
        continue
    for cusip, factor in pattern.findall(str(r['Deliverable'])):
        rows.append({
            'Contract': r['Contract'],
            'cusip': cusip,
            'conversion_factor': float(factor),
            'future_short': future_short,
            'maturity_month': maturity_month,
        })

deliverables_long = pd.DataFrame(rows)

# Now the merge works
deliv_with_isin = deliverables_long.merge(
    cusip_translation, left_on='cusip', right_on='CUSIP', how='left'
)

isin_lookup = deliv_with_isin.dropna(subset=['ISIN', 'future_short', 'maturity_month'])[
    ['ISIN', 'future_short', 'maturity_month']
].drop_duplicates()

# Remove ambiguous mappings
ambiguous = isin_lookup.groupby(['ISIN', 'maturity_month']).filter(
    lambda x: x['future_short'].nunique() > 1
)[['ISIN', 'maturity_month']].drop_duplicates()

isin_lookup_clean = isin_lookup.merge(
    ambiguous, on=['ISIN', 'maturity_month'], how='left', indicator=True
)
isin_lookup_clean = isin_lookup_clean[isin_lookup_clean['_merge'] == 'left_only'].drop(columns='_merge')

print(f"Total lookup entries: {len(isin_lookup)}")
print(f"Ambiguous removed: {len(isin_lookup) - len(isin_lookup_clean)}")
print(f"Clean lookup entries: {len(isin_lookup_clean)}")

# Step 6: Apply to main df where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')



In [ ]:
df_fix = df_emir.loc[needs_fill, ['product_isin', 'maturity_month']].reset_index().merge(
    isin_lookup_clean,
    left_on=['product_isin', 'maturity_month'],
    right_on=['ISIN', 'maturity_month'],
    how='left'
).set_index('index')

df_emir.loc[needs_fill, 'future_short'] = df_fix['future_short']

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_fix = df_emir.loc[needs_fill, ['underlying_id', 'maturity_month']].reset_index().merge(
    isin_lookup_clean,
    left_on=['underlying_id', 'maturity_month'],
    right_on=['ISIN', 'maturity_month'],
    how='left'
).set_index('index')

df_emir.loc[needs_fill, 'future_short'] = df_fix['future_short']

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

## Matching EUR deliverables

In [ ]:
deliverables_eur = pd.read_excel('Data\\deliverable_bonds.xlsx')

In [ ]:
# Step 1: Map contract name to future_short
eur_contract_map = {
    'Euro-Bund': 'bund',
    'Euro-Bobl': 'bobl',
    'Euro-Schatz': 'schatz',
    'Euro-OAT': 'oat',
    'Long-Term Euro-BTP': 'btp',
    'Euro-Buxl': 'buxl',
    'Mid-Term Euro-BTP': 'btp',  # flag this if it should be separate
    'Short-Term Euro-BTP': 'short_btp',
}

deliverables_eur['future_short'] = deliverables_eur['Contract'].map(eur_contract_map)

# Step 2: Build maturity_month from Delivery Month
deliverables_eur['maturity_month'] = pd.to_datetime(deliverables_eur['Delivery Month']).dt.strftime('%m_%y')

# Step 3: Build lookup (Deliverable Bond is the ISIN)
isin_lookup_eur = deliverables_eur.dropna(subset=['Deliverable Bond', 'future_short', 'maturity_month'])[
    ['Deliverable Bond', 'future_short', 'maturity_month']
].drop_duplicates()

# Step 4: Check for ambiguous mappings
ambiguous_eur = isin_lookup_eur.groupby(['Deliverable Bond', 'maturity_month']).filter(
    lambda x: x['future_short'].nunique() > 1
)[['Deliverable Bond', 'maturity_month']].drop_duplicates()

isin_lookup_eur_clean = isin_lookup_eur.merge(
    ambiguous_eur,
    on=['Deliverable Bond', 'maturity_month'],
    how='left',
    indicator=True
)
isin_lookup_eur_clean = isin_lookup_eur_clean[isin_lookup_eur_clean['_merge'] == 'left_only'].drop(columns='_merge')

print(f"Total lookup entries: {len(isin_lookup_eur)}")
print(f"Ambiguous removed: {len(isin_lookup_eur) - len(isin_lookup_eur_clean)}")
print(f"Clean lookup entries: {len(isin_lookup_eur_clean)}")

In [ ]:
# Step 5: Apply to main df where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')

df_fix = df_emir.loc[needs_fill, ['product_isin', 'maturity_month']].reset_index().merge(
    isin_lookup_eur_clean,
    left_on=['product_isin', 'maturity_month'],
    right_on=['Deliverable Bond', 'maturity_month'],
    how='left'
).set_index('index')

df_emir.loc[needs_fill, 'future_short'] = df_fix['future_short']

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
# Step 5: Apply to main df where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')

df_fix = df_emir.loc[needs_fill, ['underlying_id', 'maturity_month']].reset_index().merge(
    isin_lookup_eur_clean,
    left_on=['underlying_id', 'maturity_month'],
    right_on=['Deliverable Bond', 'maturity_month'],
    how='left'
).set_index('index')

df_emir.loc[needs_fill, 'future_short'] = df_fix['future_short']

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

### additional check with Davide's deliverables

In [ ]:
deliverables_cusip = pd.read_excel('Data\\Deliverables_EU.xlsx')

In [ ]:
cusip_eur_translation = pd.read_excel('Data\\EA_ISIN_TO_CUSIPS.xlsx')

In [ ]:
# Sanity check: does stripping the check digit create any ambiguity?
cusip_eur_translation['CUSIP_8'] = cusip_eur_translation['CUSIP'].str[:8]

# Check if any 8-char CUSIP maps to multiple ISINs
cusip8_check = cusip_eur_translation.groupby('CUSIP_8')['ISIN'].nunique()
ambiguous_cusip8 = cusip8_check[cusip8_check > 1]

if len(ambiguous_cusip8) > 0:
    print(f"WARNING: {len(ambiguous_cusip8)} 8-char CUSIPs map to multiple ISINs!")
    print(ambiguous_cusip8)
    # Show the problematic rows
    problem_cusips = ambiguous_cusip8.index.tolist()
    print(cusip_eur_translation[cusip_eur_translation['CUSIP_8'].isin(problem_cusips)].sort_values('CUSIP_8'))
else:
    print("All clear: every 8-char CUSIP maps to exactly one ISIN.")

In [ ]:
# Step 2: Melt deliverables_cusip into long format
deliverable_cols = [c for c in deliverables_cusip.columns if c.startswith('Deliverable')]
deliv_long = deliverables_cusip.melt(
    id_vars='Contract',
    value_vars=deliverable_cols,
    value_name='CUSIP'
).dropna(subset=['CUSIP'])[['Contract', 'CUSIP']]

print(f"Deliverables after melt: {len(deliv_long)} rows")

# Step 3: Merge with translation on 8-char CUSIP
deliv_long = deliv_long.merge(cusip_eur_translation[['CUSIP_8', 'ISIN']], left_on='CUSIP', right_on='CUSIP_8', how='left')

print(f"Matched to ISIN: {deliv_long['ISIN'].notna().sum()} / {len(deliv_long)}")

# Step 4: Parse contract -> future_short and maturity_month
eur_prefix_map = {
    'RX': 'bund', 'OE': 'bobl', 'DU': 'schatz',
    'IK': 'btp', 'OAT': 'oat', 'UB': 'buxl', 'BTS': 'short_btp',
}
month_map = {'H': '03', 'M': '06', 'U': '09', 'Z': '12'}

def parse_eur_contract(contract):
    match = re.match(r'^([A-Z]+)([HMUZ])(\d{1,2})$', contract)
    if not match:
        return None, None
    prefix = match.group(1)
    future_short = eur_prefix_map.get(prefix)
    month = month_map[match.group(2)]
    year = match.group(3)
    if len(year) == 1:
        year = '2' + year
    return future_short, f"{month}_{year}"

deliv_long[['future_short', 'maturity_month']] = deliv_long['Contract'].apply(
    lambda x: pd.Series(parse_eur_contract(x))
)

# Step 5: Build clean lookup (remove ambiguous)
isin_lookup_eur2 = deliv_long.dropna(subset=['ISIN', 'future_short', 'maturity_month'])[
    ['ISIN', 'future_short', 'maturity_month']
].drop_duplicates()

ambiguous = isin_lookup_eur2.groupby(['ISIN', 'maturity_month']).filter(
    lambda x: x['future_short'].nunique() > 1
)[['ISIN', 'maturity_month']].drop_duplicates()

isin_lookup_eur2_clean = isin_lookup_eur2.merge(
    ambiguous, on=['ISIN', 'maturity_month'], how='left', indicator=True
)
isin_lookup_eur2_clean = isin_lookup_eur2_clean[isin_lookup_eur2_clean['_merge'] == 'left_only'].drop(columns='_merge')

print(f"\nClean lookup entries: {len(isin_lookup_eur2_clean)}")
print(f"Ambiguous removed: {len(isin_lookup_eur2) - len(isin_lookup_eur2_clean)}")

In [ ]:
# Step 6: Apply to main df
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')

df_fix = df_emir.loc[needs_fill, ['product_isin', 'maturity_month']].reset_index().merge(
    isin_lookup_eur2_clean,
    left_on=['product_isin', 'maturity_month'],
    right_on=['ISIN', 'maturity_month'],
    how='left'
).set_index('index')

df_emir.loc[needs_fill, 'future_short'] = df_fix['future_short']

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
# Step 6: Apply to main df
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')

df_fix = df_emir.loc[needs_fill, ['underlying_id', 'maturity_month']].reset_index().merge(
    isin_lookup_eur2_clean,
    left_on=['underlying_id', 'maturity_month'],
    right_on=['ISIN', 'maturity_month'],
    how='left'
).set_index('index')

df_emir.loc[needs_fill, 'future_short'] = df_fix['future_short']

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

## ISINs with non-standard patterns

In [ ]:
# ISINs with non-standard pattern, looked them up individually
historical_contract_isins_2 = {
    'DE0009652645': 'bund',
    'DE0009652648': 'bund',
    'DE000C75XMU8': 'bund',
    'DE000C75XMW4': 'schatz',
    'DE000C75XMZ7': 'oat',
    'DE000C6YTCK8': 'btp',
    'DE000C6YTCM4': 'bund',
    'DE000C5GTXB2': 'bund',
    'DE000C5GTX90': 'btp',
    'DE000C5RQD86': 'btp',
    'DE000C52GUC3': 'btp',
    'DE000C52GUE9': 'bund',
    'DE000C52GUF6': 'bobl',
    'DE000C52GUG4': 'bobl',
    'DE000C52GUH2': 'buxl',
    'DE000C52GUK6': 'oat',
    'DE000C58EEE5': 'btp',
    'DE000C58EEF2': 'short_btp',
    'DE000C58EEG0': 'bund',
    'DE000C58EEH8': 'bobl',
    'DE000C58EEJ4': 'schatz',
    'DE000C58EEK2': 'buxl',
    'DE000C58EEM8': 'oat',
    'DE000C6EBRC3': 'oat',
    'DE000C6EBR57': 'btp',
    'DE000C6EBR65': 'short_btp',
    'DE000C6EBR73': 'bund',
    'DE000C6EBR81': 'bobl',
    'DE000C6EBR99': 'schatz',
    'DE000C6EBRA7': 'buxl',
    'DE000C1T6JD0': 'btp',
    'DE000C1T6JE8': 'bund',
    'DE000C1T6JF5': 'bobl',
    'DE000C1T6JG3': 'schatz',
    'DE000C1T6JH1': 'buxl',
    'DE000C1T6JK5': 'oat',
    'DE000C6J0JZ3': 'btp',
    'DE000C6J0K11': 'bund',
    'DE000C6J0K29': 'bobl',
    'DE000C6J0K37': 'schatz',
    'DE000C6J0K45': 'buxl',
    'DE000C6J0K52': 'oat',
    'DE000C6R0411': 'btp',
    'DE000C6R0452': 'bund',
    'DE000C671E06': 'btp',
    'DE000A3ETB78': 'bund',
}



In [ ]:
# Apply where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'product_isin'].map(historical_contract_isins_2)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
# Apply where still unmapped
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'underlying_id'].map(historical_contract_isins_2)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

## Flagging what is certainly not a relevant bond future

In [ ]:
# these are rate futures or things outside of the jurisdiction
df_emir.loc[(df_emir['future_short'].isna()) & (df_emir['underlying_id'].str[:2].isin(['AU', 'CA', 'JP', 'GB', 'EU'])), 'future_short'] = 'not_relevant_future'
df_emir.loc[(df_emir['future_short'].isna()) & (df_emir['product_isin'].str[:2].isin(['AU', 'CA', 'JP', 'GB', 'EU'])), 'future_short'] = 'not_relevant_future'

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

## flagging equities

In [ ]:
equities = [
    'US12572Q1058',  
    'US90390T1051', 
    'US63110F1012', 
    'US7827001089',  
    'US78378X1072',  
    'US64110L1061',  
    'US12495D1081',  
    'US74933W4942',  
    'US2605661048',  
    'US81379LAA44'  
]

In [ ]:
df_emir.loc[(df_emir['underlying_id'].isin(equities)), 'future_short'] = 'equity'

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

## Everything that is not identified by bloomberg

In [ ]:
us_futures_isins = {
    'US12573H2866': 'tn2',
    'US12573H3104': 'tbond',
    'US12573H3286': 'tn10',
    'US12573L7174': 'tn5',
}

In [ ]:
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'product_isin'].map(us_futures_isins)

df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
needs_fill = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[needs_fill, 'future_short'] = df_emir.loc[needs_fill, 'underlying_id'].map(us_futures_isins)

df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

## patterns

In [ ]:
import re

def map_remaining_isins(isin):
    if not isinstance(isin, str):
        return None
    s = isin.strip()

    # XCME - SOFR (confident)
    if s.startswith('XCME-SR3-F'):
        return 'RATE_FUTURE_REMOVE'

    # All XCBT coded identifiers -> generic US future
    if s.startswith('XCBT'):
        return 'US_future'

    # Short ticker codes (confident)
    ticker_map = {
        'ZF': 'tn5', 'ZN': 'tn10', 'ZB': 'tbond', 'TY': 'tn10',
        'UB': 'ultrabond',
    }
    if s in ticker_map:
        return ticker_map[s]

    # Short tickers (unverified)
    if s in ('UBEF', '17F', '21F'):
        return 'US_future'

    return None

# Apply to remaining unmapped rows
still_needs = df_emir['future_short'].isna() | (df_emir['future_short'] == 'NEEDS_UNDERLYING_ID')
df_emir.loc[still_needs, 'future_short'] = df_emir.loc[still_needs, 'product_isin'].apply(map_remaining_isins)

# Rebuild futures_contract
df_emir['futures_contract'] = df_emir['future_short'] + '_' + df_emir['maturity_month']

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

In [ ]:
df_emir['underlying_id'][df_emir['future_short'].isna()].value_counts(dropna = False)

In [ ]:
# this covers most of it. The remaining part is overwhelmingly observations where there is no info on the product

In [ ]:
len(df_emir[df_emir['future_short'].isna()])/len(df_emir)

In [ ]:
bond_futures = {'bund', 'bobl', 'buxl', 'schatz', 'oat', 'bono', 'btp', 'short_btp',
                'tn2', 'tn3', 'tn5', 'tn10', 'tbond', 'ultrabond', 'ultra10', 'US_future'}

df_emir['is_bond_future'] = df_emir['future_short'].isin(bond_futures).astype(int)
df_emir['futures_contract'] = None
df_emir.loc[df_emir['is_bond_future'] == 1, 'futures_contract'] = (
    df_emir.loc[df_emir['is_bond_future'] == 1, 'future_short'] + '_' + 
    df_emir.loc[df_emir['is_bond_future'] == 1, 'maturity_month']
)

In [ ]:
df_emir['vol'][df_emir['future_short'].isna()].sum()/df_emir['vol'].sum()

In [ ]:
df_emir['vol'][df_emir['futures_contract'].isna()].sum()/df_emir['vol'].sum()

In [ ]:
df_emir['is_bond_future'].value_counts()

In [ ]:
df_emir[(df_emir['product_isin'].isna()) & (df_emir['future_short'].str[:3].isin(['bob', 'sch', 'bun', 'bux']))].sample(5)

In [ ]:
df_emir['vol'][(df_emir['product_isin'].isna()) & (df_emir['future_short'].str[:3].isin(['bob', 'sch', 'bun', 'bux']))].sum()/df_emir['vol'][(df_emir['future_short'].str[:3].isin(['bob', 'sch', 'bun', 'bux']))].sum()

In [ ]:
pd.DataFrame(df_emir['product_isin'][df_emir['future_short'].str[:3].isin(['bob', 'sch', 'bun', 'bux'])].unique()).to_csv('Data\\german_futures.csv')

## Creating final dataset

In [ ]:
# Map direction to hedge fund perspective
direction_map = {
    'BYER': 'short_futures',
    'TAKE': 'short_futures',
    'SLLR': 'long_futures',
    'MAKE': 'long_futures',
}
df_emir['hf_direction'] = df_emir['direction'].map(direction_map)

df_emir['futures_contract'] = df_emir['futures_contract'].fillna('not_an_identified_bond_future')

df_pivot = df_emir.pivot_table(
    index=['business_date', 'entity_id', 'bank_indicator', 'currency', 'future_short', 'futures_contract', 'maturity_month', 'is_bond_future'],
    columns='hf_direction',
    values='vol',
    aggfunc='sum',
    fill_value=0
).reset_index()

In [ ]:
df_pivot['is_bond_future'].value_counts()

In [ ]:
df_pivot.head()

In [ ]:
#df_pivot.to_csv('emir_dataframe.csv')

In [ ]:
#df_emir = pd.read_csv('emir_dataframe.csv')

In [ ]:
len(df_pivot['entity_id'].unique())

In [ ]:
df_pivot = df_pivot[['business_date', 'entity_id', 'bank_indicator', 'currency',
       'futures_contract', 'maturity_month', 'is_bond_future', 'long_futures',
       'short_futures']]

In [ ]:
df_pivot.loc[(df_pivot['entity_id'].isin(hf_overlap)), 'sftds_overlap'] = 1
df_pivot['sftds_overlap'].fillna(0, inplace = True)

In [ ]:
df_pivot[df_pivot['sftds_overlap'] == 1].head()

In [ ]:
df_pivot['bank_indicator'].value_counts()

In [ ]:
#df_emir.to_csv('emir_dataframe.csv')

In [ ]:
#df_emir = pd.read_csv('emir_dataframe.csv', index_col= 0)

In [ ]:
us_prefix_map = {
    'TY': 'tn10',
    'FV': 'tn5',
    'TU': 'tn2',
    'UXY': 'ultra10',
    'WN': 'ultrabond',
    'US': 'tbond',
}

In [ ]:
eur_prefix_map = {
    'RX': 'bund', 'OE': 'bobl', 'DU': 'schatz',
    'IK': 'btp', 'OAT': 'oat', 'UB': 'buxl', 'BTS': 'short_btp',
}

In [ ]:
# Invert the maps to go from suffix -> bloomberg prefix
us_inv = {v: k for k, v in us_prefix_map.items()}
eur_inv = {v: k for k, v in eur_prefix_map.items()}
prefix_map = {**us_inv, **eur_inv}
month_map = {'H': '03', 'M': '06', 'U': '09', 'Z': '12'}

month_to_letter = {v: k for k, v in month_map.items()}

def to_bloomberg(fc):
    parts = fc.split('_')
    if len(parts) < 3:
        return None
    name = '_'.join(parts[:-2])
    month = parts[-2]
    year = parts[-1]
    prefix = prefix_map.get(name)
    month_letter = month_to_letter.get(month)
    if prefix is None or month_letter is None:
        return None
    return prefix + month_letter + year

df_pivot['futures_identifier'] = df_pivot['futures_contract'].apply(to_bloomberg)

In [ ]:
df_pivot = df_pivot.rename_axis(columns=None)

In [ ]:
df_pivot.sample(5)

In [ ]:
df_pivot.columns

In [ ]:
df_pivot.to_csv('key dataframe\\emir_dataframe.csv')

# Bond day panel, bond_day.csv

In [ ]:
df = pd.read_excel('Data\\EA_ISINs.xlsx')

In [ ]:
cusip_eur_translation = pd.read_excel('Data\\EA_ISIN_TO_CUSIPS.xlsx')

In [ ]:
df.head()

In [ ]:
df = df.merge(cusip_eur_translation, on = 'ISIN', how = 'left')

In [ ]:
mts = pd.read_csv('Data\\mts.csv', index_col= 0)

In [ ]:
mts.head()

In [ ]:
df = df.merge(mts, left_on = 'ISIN', right_on = 'bondcode', how = 'outer')

In [ ]:
df = df[['refdate', 'ISIN', 'CUSIP', 'bondtype', 'country', 'description',
       'issuertype', 'issuedate', 'maturitydate', 'coupontype', 'couponrate',
       'couponfreq', 'firstcoupondate', 'lastcoupondate', 'currency',
       'firstsettldate', 'priceorigin', 'refprice', 'refyield',
       'strips', 'settldate']]

In [ ]:
df_us = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [ ]:
df_us.head()

In [ ]:
us_prices = pd.read_csv('NelsonSiegel//NelsonSiegel_US_prices.csv')

In [ ]:
us_prices = us_prices[['issuedate', 'maturitydate', 'yield', 'TCUSIP',
       'settldate', 'refdate', 'couponrate',
       'firstcoupondate', 'country',
       'refprice', 'couponfreq', 'lastcoupondate']]

In [ ]:
df_us = df_us.merge(us_prices, left_on = 'CUSIP', right_on = 'TCUSIP', how = 'right')

In [ ]:
df_us.head()

In [ ]:
df.columns

In [ ]:
df = df[['refdate', 'ISIN', 'CUSIP', 'country', 'issuedate', 'maturitydate', 'couponrate',
       'couponfreq', 'firstcoupondate', 'lastcoupondate',
       'refprice', 'refyield',
       'settldate']]

In [ ]:
df['yield'] = df['refyield']

In [ ]:
df = df[['refdate', 'ISIN', 'CUSIP', 'country', 'issuedate', 'maturitydate', 'couponrate',
       'couponfreq', 'firstcoupondate', 'lastcoupondate',
       'refprice', 'yield',
       'settldate']]

In [ ]:
df_us = df_us[['refdate', 'ISIN', 'CUSIP', 'country', 'issuedate', 'maturitydate', 'couponrate', 'couponfreq', 'firstcoupondate', 'lastcoupondate',
                'refprice', 'yield', 'settldate']]

In [ ]:
df = pd.concat([df, df_us])

In [ ]:
# --- 1. Parse EUBASIS.xlsx into long format ---
df_raw = pd.read_excel('Data\\EUBASIS.xlsx', header=None)

In [ ]:
products = [
    (1, 'RX1'), (6, 'RX2'), (11, 'OE1'), (16, 'OE2'),
    (21, 'DU1'), (26, 'DU2'), (31, 'IK1'), (36, 'IK2'),
    (41, 'OAT1'), (46, 'OAT2'), (51, 'UB1'), (56, 'UB2'),
    (61, 'BTS1'), (66, 'BTS2')
]

data = df_raw.iloc[3:].copy()
data[0] = pd.to_datetime(data[0])

frames = []
for start_col, label in products:
    block = data[[0, start_col, start_col+1, start_col+2, start_col+3, start_col+4]].copy()
    block.columns = ['date', 'implied_repo', 'open_interest', 'ctd_net_basis', 'ticker', 'ctd_cusip']
    frames.append(block)

df_basis = pd.concat(frames, ignore_index=True)

In [ ]:
# --- Parse USBASIS.csv into long format ---
df_us_raw = pd.read_csv('Data\\USBASIS.csv')
df_us_raw['Date'] = pd.to_datetime(df_us_raw['Date'], format='%m/%d/%Y')

us_products = ['TY1', 'TY2', 'FV1', 'FV2', 'TU1', 'TU2',
               'UXY1', 'UXY2', 'WN1', 'WN2', 'US1', 'US2']

us_ticker_to_category = {
    'UXY': 'ultra10', 'WN': 'ultrabond', 'TY': 'tn10',
    'FV': 'tn5', 'TU': 'tn2', 'US': 'tbond'
}

def ticker_to_fc_us(ticker):
    if pd.isna(ticker):
        return None
    for prefix, cat in sorted(us_ticker_to_category.items(), key=lambda x: -len(x[0])):
        if ticker.startswith(prefix):
            rest = ticker[len(prefix):]
            return f"{cat}_{month_map.get(rest[0])}_{rest[1:]}"
    return None

us_frames = []
for prod in us_products:
    block = df_us_raw[['Date',
                        f'{prod}_FUT_IMPLIED_REPO_RT',
                        f'{prod}_OPEN_INT',
                        f'{prod}_FUT_CTD_NET_BASIS',
                        f'{prod}_FUT_CUR_GEN_TICKER',
                        f'{prod}_FUT_CTD_CUSIP']].copy()
    block.columns = ['date', 'implied_repo', 'open_interest', 'ctd_net_basis', 'ticker', 'ctd_cusip']
    us_frames.append(block)

df_basis_us = pd.concat(us_frames, ignore_index=True)

In [ ]:
# --- Combine EU and US basis data ---
df_basis = pd.concat([df_basis, df_basis_us], ignore_index=True)

In [ ]:
df_basis.head()

In [ ]:
df['CUSIP'] = df['CUSIP'].str[:8]

In [ ]:
df['refdate'] = pd.to_datetime(df['refdate'])
df_basis['date'] = pd.to_datetime(df_basis['date'])

In [ ]:
df = df.merge(df_basis, left_on = ['refdate', 'CUSIP'], right_on = ['date', 'ctd_cusip'], how = 'left')

In [ ]:
df.columns

In [ ]:
df[['ctd_implied_repo', 'ctd_open_interest', 'ctd_ticker']] = df[['implied_repo', 'open_interest', 'ticker']]

In [ ]:
df = df[['refdate', 'ISIN', 'CUSIP', 'country', 'issuedate', 'maturitydate',
       'couponrate', 'couponfreq', 'firstcoupondate', 'lastcoupondate',
       'refprice', 'yield', 'settldate', 'implied_repo',
       'open_interest', 'ctd_net_basis', 'ticker',
       'ctd_implied_repo', 'ctd_open_interest', 'ctd_ticker']]

In [ ]:
df.to_csv('key dataframe\\bond_day.csv')

In [ ]:
df = pd.read_csv('key dataframe\\bond_day.csv', index_col=0)

In [ ]:
df.columns

In [ ]:
df = df[['refdate', 'ISIN', 'CUSIP', 'country', 'issuedate', 'maturitydate',
       'couponrate', 'couponfreq', 'firstcoupondate', 'lastcoupondate',
       'refprice', 'yield', 'settldate']]

In [ ]:
df_otr = pd.read_csv('Data\\otrnumber.csv')

In [ ]:
df_otr['date'] = pd.to_datetime(df_otr['date'], dayfirst=True)
df['refdate'] = pd.to_datetime(df['refdate'], format='%Y-%m-%d')

In [ ]:
df = df.merge(df_otr[['date', 'cusip', 'otr_maturity', 'otr_type', 'otrnumber']], left_on=['refdate', 'CUSIP'], right_on=['date', 'cusip'], how = 'left')

In [ ]:
df.to_csv('key dataframe\\bond_day.csv')

# Bond level positions with Nelson Siegel prices, hf_positions.csv and hf_positions_US.csv

## For EUR

In [ ]:
# Data prep
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, SUM(nominal_value) as borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND borrower_country_residence = 'KY'
AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}
GROUP BY business_date, borrower_id, security_isin
ORDER BY business_date, borrower_id, security_isin

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, SUM(nominal_value) as lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND lender_country_residence = 'KY'
AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}
GROUP BY business_date, lender_id, security_isin
ORDER BY business_date, lender_id, security_isin

"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df = df_borrowing.merge(df_lending, on= ['business_date', 'entity_id', 'security_isin'], how = 'outer')

In [ ]:
len(df['entity_id'].unique())

In [ ]:
len(df)

In [ ]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [ ]:
df['borrowing_volume'].fillna(0,inplace=True)
df['lending_volume'].fillna(0,inplace=True)

In [ ]:
germany = pd.read_csv('NelsonSiegel_DE_prices.csv')

In [ ]:
italy = pd.read_csv('NelsonSiegel_IT_prices.csv')

In [ ]:
france = pd.read_csv('NelsonSiegel_FR_prices.csv')

In [ ]:
spain = pd.read_csv('NelsonSiegel_ES_prices.csv')

In [ ]:
df_prices = pd.concat([germany, italy, france, spain], ignore_index=True)

In [ ]:
df_prices['refdate'] = pd.to_datetime(df_prices['refdate'])

In [ ]:
len(df)

In [ ]:
outstanding = pd.read_excel('outstanding.xlsx')

In [ ]:
df = df.merge(df_prices, left_on = ['business_date', 'security_isin'], right_on = ['refdate', 'bondcode'], how = 'inner')

In [ ]:
len(df)

In [ ]:
df = df.merge(outstanding, left_on = ['security_isin'], right_on = ['ISIN'], how = 'inner')

In [ ]:
len(df)

In [ ]:
df.to_csv('Data\\hf_positions.csv')

## For USD

In [ ]:
# Data prep
query = f"""

SELECT business_date, borrower_id as entity_id, security_isin, SUM(nominal_value) as borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}
GROUP BY business_date, borrower_id, security_isin
ORDER BY business_date, borrower_id, security_isin

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT business_date, lender_id as entity_id, security_isin, SUM(nominal_value) as lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}
GROUP BY business_date, lender_id, security_isin
ORDER BY business_date, lender_id, security_isin

"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df = df_borrowing.merge(df_lending, on= ['business_date', 'entity_id', 'security_isin'], how = 'outer')

In [ ]:
pd.DataFrame(df['entity_id'].unique()).to_excel('hf_list.xlsx')

In [ ]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [ ]:
df['borrowing_volume'].fillna(0,inplace=True)
df['lending_volume'].fillna(0,inplace=True)

In [ ]:
df_prices = pd.read_csv('NelsonSiegel_US_prices.csv')

In [ ]:
df_prices['refdate'] = pd.to_datetime(df_prices['refdate'])

In [ ]:
df_prices = df_prices.merge(treasury, left_on = ['TCUSIP'], right_on = ['CUSIP'], how = 'left')

In [ ]:
df = df.merge(df_prices, left_on = ['business_date', 'security_isin'], right_on = ['refdate', 'ISIN'], how = 'inner')

In [ ]:
df.to_csv('Data\\hf_positions_US.csv')

# Hedge fund country list, fund_countries.csv

## produce entity country list

In [ ]:
df = pd.read_csv('key dataframe\\emir_dataframe.csv')

In [ ]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [ ]:
hf_overlap = tuple(hedge_funds['entity_id'].unique())

In [ ]:
# Data prep
query = f"""


WITH country_counts AS (
    SELECT 
        other_cpty_id AS entity_id,
        other_cpty_country AS fund_country,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (
            PARTITION BY other_cpty_id 
            ORDER BY COUNT(*) DESC, other_cpty_country
        ) AS rn
    FROM crp_emir_ecb.emir_ecb_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
    WHERE reference_period >= '2021-01-04'
    AND reference_period <= '2024-04-29'
    AND notional < 1e18
    AND asset_class = 'INTR'
    AND contract_type = 'FUTR'
    AND notional > 0
    AND notional_currency1 IN ("EUR", "USD")
    AND LEFT(product_clssfctn, 3) = 'FFD'
    AND ((LEFT(underlying_id, 2) NOT IN ('EU', 'XC')) OR underlying_id IS NULL)
    AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
    AND MONTH(maturity_date) IN (3, 6, 9, 12)
    AND (s_entity.sector = 'IF' OR s_entity.sector = 'OFI' OR s_entity.sector = 'OTHR' OR other_cpty_id_esa_sector = 'S124' OR other_cpty_id IN {hf_overlap})
    AND other_cpty_country IS NOT NULL
    GROUP BY other_cpty_id, other_cpty_country
)
SELECT entity_id, fund_country
FROM country_counts
WHERE rn = 1


"""

df_emir_old = pd.read_sql_query(query, cnxn)

In [ ]:
df_emir_old.head()

In [ ]:
# Data prep
query = f"""


WITH country_counts AS (
    SELECT 
        other_cpty_id AS entity_id,
        other_cpty_country AS fund_country,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (
            PARTITION BY other_cpty_id 
            ORDER BY COUNT(*) DESC, other_cpty_country
        ) AS rn
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
    WHERE reference_period >= '2024-04-29'
    AND reference_period <= '2025-12-31'
    AND notional_leg1 < 1e18
    AND asset_class = 'INTR'
    AND contract_type = 'FUTR'
    AND notional_leg1 > 0
    AND notional_leg1_currency IN ("EUR", "USD")
    AND LEFT(product_cfi, 3) = 'FFD'
    AND (LEFT(underlying_id, 2) != 'EU' OR underlying_id IS NULL)
    AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
    AND MONTH(maturity_date) IN (3, 6, 9, 12)
    AND (s_entity.sector = 'IF' OR other_cpty_id_esa_sector_riad = 'S124' OR other_cpty_id IN {hf_overlap})
    AND other_cpty_country IS NOT NULL
    GROUP BY other_cpty_id, other_cpty_country
)
SELECT entity_id, fund_country
FROM country_counts
WHERE rn = 1


"""

df_emir_refit = pd.read_sql_query(query, cnxn)

In [ ]:
df_emir_refit.head()

In [ ]:
df_emir_old['fund_country'].value_counts()

In [ ]:
df_emir_refit['fund_country'].value_counts()

In [ ]:
import pandas as pd

# Combine both sources, tracking origin for debugging
combined = pd.concat([
    df_emir_old.assign(source='old'),
    df_emir_refit.assign(source='refit')
], ignore_index=True)

# Find entities with conflicting countries across the two sources
conflicts = (
    combined.groupby('entity_id')['fund_country']
    .nunique()
    .loc[lambda s: s > 1]
    .index
)

print(f"Entities with conflicting countries: {len(conflicts)}")
if len(conflicts) > 0:
    print(combined[combined['entity_id'].isin(conflicts)].sort_values('entity_id'))

In [ ]:
# Option C: prefer refit, but fall back to old if refit is missing
df_final = (
    df_emir_refit
    .set_index('entity_id')
    .combine_first(df_emir_old.set_index('entity_id'))
    .reset_index()
)


In [ ]:
df_final.head()

In [ ]:
len(df)

In [ ]:
df = df.merge(df_final, on = 'entity_id', how = 'left')

In [ ]:
len(df)

In [ ]:
df.head()

In [ ]:
df['fund_country'].fillna('Other', inplace = True)

In [ ]:
df['fund_country'].value_counts(dropna=False)

In [ ]:
sftds_hf = pd.read_csv("key dataframe\\sftds_hedgefunds.csv")

In [ ]:
sftds_entities = tuple(sftds_hf['0'].unique())

In [ ]:
# data prep
query = f"""

WITH borrower_side AS (
    SELECT 
        s.borrower_id AS entity_id, 
        s.borrower_country_residence AS fund_country
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
    WHERE s.nominal_ccy IN ('EUR', 'USD')
      AND s.business_date <= '2025-10-01'
      AND s.borrower_id IN {sftds_entities}
      AND s.borrower_country_residence IS NOT NULL
),
lender_side AS (
    SELECT 
        s.lender_id AS entity_id, 
        s.lender_country_residence AS fund_country
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
    WHERE s.nominal_ccy IN ('EUR', 'USD')
      AND s.business_date <= '2025-10-01'
      AND s.lender_id IN {sftds_entities}
      AND s.lender_country_residence IS NOT NULL
),
combined AS (
    SELECT entity_id, fund_country FROM borrower_side
    UNION ALL
    SELECT entity_id, fund_country FROM lender_side
),
country_counts AS (
    SELECT 
        entity_id,
        fund_country,
        ROW_NUMBER() OVER (
            PARTITION BY entity_id 
            ORDER BY COUNT(*) DESC, fund_country
        ) AS rn
    FROM combined
    GROUP BY entity_id, fund_country
)
SELECT entity_id, fund_country
FROM country_counts
WHERE rn = 1

"""

df_sftds = pd.read_sql_query(query, cnxn)

In [ ]:
df_sftds.head()

In [ ]:
df_sftds_full = pd.read_csv("key dataframe\\sftds_dataframe.csv")

In [ ]:
df_sftds_full = df_sftds_full.merge(df_sftds, on = 'entity_id', how = 'left')

In [ ]:
df_sftds_full['fund_country'].fillna('Other', inplace = True)

In [ ]:
sftds_combination = df_sftds_full[['entity_id', 'fund_country']].drop_duplicates()

In [ ]:
emir_combination = df[['entity_id', 'fund_country']].drop_duplicates()

In [ ]:
combined = (
    pd.concat([sftds_combination, emir_combination], ignore_index=True)
    .drop_duplicates(subset='entity_id', keep='first')
    .reset_index(drop=True)
)

In [ ]:
combined.to_csv('key dataframe\\fund_countries.csv', index = False)

# Fund locations, germany_fund_location.csv and us_fund_location.csv

## Germany

In [ ]:
df = pd.read_csv('key dataframe\\emir_dataframe.csv', index_col= 0)

In [ ]:
countries = pd.read_csv('key dataframe\\fund_countries.csv')

In [ ]:
df = df.merge(countries, on = 'entity_id', how = 'left')

In [ ]:
df_de = df[(df['is_bond_future'] == 1) & (df['currency'] == 'EUR') & (df['futures_contract'].str[:3].isin(['bun', 'sch', 'bob', 'bux'])) & (df['long_futures'] <= 1e10) & (df['short_futures'] <= 1e10)]

In [ ]:
df_de['fund_country'].fillna('Other', inplace = True)

In [ ]:
df_de = df_de[['business_date', 'fund_country', 'long_futures', 'short_futures']].groupby(['business_date', 'fund_country']).sum().reset_index()

In [ ]:
df_de['net_pos'] = (df_de['long_futures'] - df_de['short_futures'])/1e9

In [ ]:
df_de['business_date'] = pd.to_datetime(df_de['business_date'])

In [ ]:
# Euro area country codes (ISO 3166-1 alpha-2)
EA_COUNTRIES = {
    'AT', 'BE', 'CY', 'DE', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR',
    'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PT', 'SI', 'SK'
}

# Countries shown individually
NAMED = ['DE', 'FR', 'IE', 'LU', 'GB', 'KY']

def bucket(country):
    if country in NAMED:
        return country
    if country in EA_COUNTRIES:
        return 'Other EA'
    return 'Other non-EA'

df_de = df_de.copy()
df_de['group'] = df_de['fund_country'].map(bucket)

In [ ]:
df_de.to_csv('Data\\germany_fund_location.csv')

## US

In [ ]:
df = pd.read_csv('key dataframe\\emir_dataframe.csv', index_col= 0)

In [ ]:
countries = pd.read_csv('key dataframe\\fund_countries.csv')

In [ ]:
df = df.merge(countries, on = 'entity_id', how = 'left')

In [ ]:
df_us = df[(df['is_bond_future'] == 1) & (df['currency'] == 'USD') & (df['long_futures'] <= 1e10) & (df['short_futures'] <= 1e10)]

In [ ]:
df_us['fund_country'].fillna('Other', inplace = True)

In [ ]:
df_us = df_us[['business_date', 'fund_country', 'long_futures', 'short_futures']].groupby(['business_date', 'fund_country']).sum().reset_index()

In [ ]:
df_us['net_pos'] = (df_us['long_futures'] - df_us['short_futures'])/1e9

In [ ]:
df_us['business_date'] = pd.to_datetime(df_us['business_date'])

In [ ]:
# Euro area country codes (ISO 3166-1 alpha-2)
EA_COUNTRIES = {
    'AT', 'BE', 'CY', 'DE', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR',
    'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PT', 'SI', 'SK'
}

# Countries shown individually
NAMED = ['DE', 'FR', 'IE', 'LU', 'GB', 'KY']

def bucket(country):
    if country in NAMED:
        return country
    if country in EA_COUNTRIES:
        return 'Other EA'
    return 'Other non-EA'

df_us['group'] = df_us['fund_country'].map(bucket)

In [ ]:
df_us.to_csv('Data\\us_fund_location.csv')

# Net futures and repo positions of UK banks and Cayman funds, netpositions_limitations csv files

## Aggregated

In [ ]:
# Data prep
query = f"""

SELECT reference_period as business_date, notional_leg1_currency as currency, CASE WHEN direction = 'MAKE' THEN 'SLLR' WHEN direction = 'TAKE' THEN 'BYER' ELSE direction END AS direction, SUM(notional_leg1) as vol
FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
WHERE reference_period >= '2024-04-29'
AND reference_period <= '2025-12-31'
AND notional_leg1 < 1e18
AND asset_class = 'INTR'
AND contract_type = 'FUTR'
AND notional_leg1 > 0
AND notional_leg1_currency IN ("EUR")
AND LEFT(product_cfi, 3) = 'FFD'
AND (LEFT(underlying_id, 2) != 'EU' OR underlying_id IS NULL)
AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
AND MONTH(maturity_date) IN (3, 6, 9, 12)
AND (((s_entity.sector = 'BANK' OR s_entity.sector = 'OFI') AND other_cpty_id_country_code_gleif = 'GB') OR other_cpty_id_country_code_gleif = 'KY')
AND ccp_id = '529900LN3S50JPU47S06'
AND notional_leg1 <= 5e10
GROUP BY reference_period, currency, direction
ORDER BY reference_period, currency, direction

"""

df_emir_refit = pd.read_sql_query(query, cnxn)

In [ ]:
# 94% of volumes are accounted for by the case when eurex is not just the CCP but also the reporter
1445798308486950/1542281038818838

In [ ]:
# Data prep
query = f"""


SELECT reference_period as business_date, notional_currency1 as currency,  counterparty_side as direction, SUM(notional) as vol
FROM crp_emir_ecb.emir_ecb_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
WHERE reference_period >= '2021-01-04'
AND reference_period <= '2024-04-29'
AND notional < 1e18
AND asset_class = 'INTR'
AND contract_type = 'FUTR'
AND notional > 0
AND notional_currency1 IN ("EUR")
AND LEFT(product_clssfctn, 3) = 'FFD'
AND ((LEFT(underlying_id, 2) NOT IN ('EU', 'XC')) OR underlying_id IS NULL)
AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
AND MONTH(maturity_date) IN (3, 6, 9, 12)
AND (((s_entity.sector = 'BANK' OR s_entity.sector = 'OFI') AND other_cpty_id_country_code = 'GB') OR other_cpty_id_country_code = 'KY')
AND ccp_id = '529900LN3S50JPU47S06'
AND notional <= 5e10
GROUP BY reference_period, currency, direction
ORDER BY reference_period, currency, direction


"""

df_emir_old = pd.read_sql_query(query, cnxn)

In [ ]:
df_emir = pd.concat([df_emir_refit, df_emir_old]).drop_duplicates().reset_index(drop=True)

In [ ]:
df_pivoted = df_emir.pivot_table(
    index=['business_date', 'currency'],
    columns='direction',
    values='vol',
    aggfunc='sum',
    fill_value=0
).reset_index()

df_pivoted.columns.name = None  # remove the 'direction' label on columns
df_pivoted = df_pivoted.rename(columns={'SLLR': 'vol_sell', 'BYER': 'vol_buy'})

In [ ]:
# data prep
query = f"""

SELECT 
    s.business_date,
    SUM(s.nominal_value) AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND ((s_borrower.sector = 'BANK' AND borrower_country = 'GB') OR borrower_country_residence = 'KY')
    AND security_type = 'GOVS'
    AND LEFT(security_isin, 2) IN ('DE', 'IT', 'ES', 'FR')
    AND assttp_scty_issr_sector_riad = 'S1311'
GROUP BY s.business_date
ORDER BY s.business_date

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    SUM(s.nominal_value) AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.nominal_ccy IN ('EUR', 'USD')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND ((s_lender.sector = 'BANK' AND lender_country = 'GB') OR lender_country_residence = 'KY')
    AND security_type = 'GOVS'
    AND LEFT(security_isin, 2) IN ('DE', 'IT', 'ES', 'FR')
    AND assttp_scty_issr_sector_riad = 'S1311'
GROUP BY s.business_date
ORDER BY s.business_date
"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df = df_borrowing.merge(df_lending, on= ['business_date'], how = 'outer')

In [ ]:
df_pivoted['business_date'] = pd.to_datetime(df_pivoted['business_date'])

In [ ]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [ ]:
df_full = df.merge(df_pivoted[['business_date', 'vol_sell', 'vol_buy']], on = 'business_date', how = 'left')

In [ ]:
df_full['net_repo'] = (df_full['borrowing_volume'] - df_full['lending_volume'])/1e9

In [ ]:
df_full['net_futures'] = (df_full['vol_sell'] - df_full['vol_buy'])/1e9

In [ ]:
df_full.tail()

In [ ]:
df_full[['business_date', 'net_futures', 'net_repo']].to_csv('Data\\broadest_netpositions_limitations.csv', index = False)

### only for germany

In [ ]:
index_names = tuple(['EURX ESCHATZ', 'EURO SCHATZ', 'Euro Schatz', 'Euro Schatz-Future',
    'Euro-Schatz', 'EURO - SCHATZ', 'Euro Schatz -2 Year Bond',
    'Euro-Schatz-2-Year-Bond',
    'EURX EURBOBL', 'EURO BOBL', 'Euro Bobl -5 Year Bond', 'Euro-Bobl',
    'Euro-Bobl Futures (FGBM)', 'Euro-Bobl Futures',
    'Euro-Bobl-5-Year-Bond', 'Euro-Bobl 5yr 6%',
    'EURO BUND', '"EURX EUR-BUND"', 'Euro Bund -10 Year Bond', 'Euro-Bund',
    'Euro-Bund Futures', 'Euro bund', 'Euro Bund', 'BUND', 'Euro Bund-Future',
    'FGBL', 'EURO BUND FUTURES', 'EURO EU BOND EUROPEAN LT UND',
    'BUND FUTURES SEP24', 'BUND FUTURES DEC24', 'BUND FUTURES MAR25',
    'BUND FUTURES JUN25', 'FGBL MAR25 [FIXED INCOME FUTURES',
    'FGBL SI 20250606 PS', 'FGBL SI 20250908 PS', 'FGBL SI 20251208 PS',
    'Euro-Bund-10-Year-Bond', 'Euro-Bund 10yr 6%',
    '"EURX EURO-BUXL"', 'GERMAN GOVERNMENT BUXL  Bond',
    'Euro Buxl -15 - 30 Year Bond', 'Euro-Buxl Futures',
    'Euro Buxl Futures', 'Euro Buxl', 'Euro-Buxl',
    'Euro-Buxl-15-30-Year-Bond'])

In [ ]:
product_isin = tuple([
    'DE0009652669','DE0009652651','DE0009652644','DE0009652636',

    'DE000F2MGDC1','DE000F2Y2EY1','DE000F3A9ET3',
    'DE000F1B2NH5','DE000F1NGF61','DE000F1ZLHQ6','DE000F2MGDD9','DE000F2Y2EZ8','DE000F3A9EU1',
    'DE000F1B2NJ1','DE000F1NGF79','DE000F1ZLHR4','DE000F2MGDE7','DE000F2Y2E04','DE000F3A9EV9',
    'DE000F1B2NK9','DE000F1NGF87','DE000F1ZLHS2','DE000F2MGDF4','DE000F2Y2E12','DE000F3A9EW7',

    'DE000F0FSQN3','DE000F0FSQP8','DE000F0FSQQ6','DE000F0Q7BW2',
    'DE000F0Q7BV4','DE000F0Q7BU6','DE000F01NAD9','DE000F01NAF4',
    'DE000F1B2NG7','DE000F1NGF53','DE000F1ZLHP8',

    'DE0009652645','DE0009652648','DE000C75XMU8','DE000C75XMW4',
    'DE000C6YTCM4','DE000C5GTXB2','DE000C52GUE9',
    'DE000C52GUF6','DE000C52GUG4','DE000C52GUH2',
    'DE000C58EEG0','DE000C58EEH8','DE000C58EEJ4','DE000C58EEK2',
    'DE000C6EBR73','DE000C6EBR81','DE000C6EBR99','DE000C6EBRA7',
    'DE000C1T6JE8','DE000C1T6JF5','DE000C1T6JG3','DE000C1T6JH1',
    'DE000C6J0K11','DE000C6J0K29','DE000C6J0K37','DE000C6J0K45',
    'DE000C6R0452','DE000A3ETB78',

    'DE000C75XMV6','DE000C5GTXC0','DE000C5RQDB7','DE000C6R0460','DE000C6YTCN2','DE000C671E30',
    'DE000C7F2YB2','DE000C7PB9Z2','DE000C7X7UL6',

    'DE000C75XMX2','DE000F0FSQR4','DE000F01NAG2','DE000C5GTXE6','DE000C5RQDD3','DE000C6R0494',
    'DE000C6YTCQ5','DE000C671E55','DE000C7F2YD8','DE000C7PB918','DE000C7X7UN2',

    'DE000BU22049','DE000C5GTXD8','DE000C5RQDC5','DE000C6R0478','DE000C6YTCP7','DE000C671E48',
    'DE000C7F2YC0','DE000C7PB900','DE000C7X7UM4','DE000F01NAE7',

    'DE000BU2Z007','DE000C5RQDA9','DE000C671E22','DE000C7F2YA4','DE000C7PB9Y5','DE000C7X7UK8',
    'DE000F1NGF20','DE0009664581'
])

In [ ]:
# Data prep
query = f"""

SELECT reference_period as business_date, notional_leg1_currency as currency, CASE WHEN direction = 'MAKE' THEN 'SLLR' WHEN direction = 'TAKE' THEN 'BYER' ELSE direction END AS direction, SUM(notional_leg1) as vol
FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
WHERE reference_period >= '2024-04-29'
AND reference_period <= '2025-12-31'
AND notional_leg1 < 1e18
AND asset_class = 'INTR'
AND contract_type = 'FUTR'
AND notional_leg1 > 0
AND notional_leg1_currency IN ("EUR")
AND LEFT(product_cfi, 3) = 'FFD'
AND (LEFT(underlying_id, 2) != 'EU' OR underlying_id IS NULL)
AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
AND MONTH(maturity_date) IN (3, 6, 9, 12)
AND (((s_entity.sector = 'BANK' OR s_entity.sector = 'OFI') AND other_cpty_id_country_code_gleif = 'GB') OR other_cpty_id_country_code_gleif = 'KY')
AND ccp_id = '529900LN3S50JPU47S06'
AND notional_leg1 <= 5e10
AND (underlying_id IN {product_isin} OR product_isin IN {product_isin} OR underlying_index_name IN {index_names})
GROUP BY reference_period, currency, direction
ORDER BY reference_period, currency, direction

"""

df_emir_refit = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""


SELECT reference_period as business_date, notional_currency1 as currency,  counterparty_side as direction, SUM(notional) as vol
FROM crp_emir_ecb.emir_ecb_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
WHERE reference_period >= '2021-01-04'
AND reference_period <= '2024-04-29'
AND notional < 1e18
AND asset_class = 'INTR'
AND contract_type = 'FUTR'
AND notional > 0
AND notional_currency1 IN ("EUR")
AND LEFT(product_clssfctn, 3) = 'FFD'
AND ((LEFT(underlying_id, 2) NOT IN ('EU', 'XC')) OR underlying_id IS NULL)
AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
AND MONTH(maturity_date) IN (3, 6, 9, 12)
AND (((s_entity.sector = 'BANK' OR s_entity.sector = 'OFI') AND other_cpty_id_country_code = 'GB') OR other_cpty_id_country_code = 'KY')
AND ccp_id = '529900LN3S50JPU47S06'
AND notional <= 5e10
AND (underlying_id IN {product_isin} OR product_id IN {product_isin} OR underlying_index_name IN {index_names})
GROUP BY reference_period, currency, direction
ORDER BY reference_period, currency, direction


"""

df_emir_old = pd.read_sql_query(query, cnxn)

In [ ]:
df_emir = pd.concat([df_emir_refit, df_emir_old]).drop_duplicates().reset_index(drop=True)

In [ ]:
df_pivoted = df_emir.pivot_table(
    index=['business_date', 'currency'],
    columns='direction',
    values='vol',
    aggfunc='sum',
    fill_value=0
).reset_index()

df_pivoted.columns.name = None  # remove the 'direction' label on columns
df_pivoted = df_pivoted.rename(columns={'SLLR': 'vol_sell', 'BYER': 'vol_buy'})

In [ ]:
# data prep
query = f"""

SELECT 
    s.business_date,
    SUM(s.nominal_value) AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND ((s_borrower.sector = 'BANK' AND borrower_country = 'GB') OR borrower_country_residence = 'KY')
    AND security_type = 'GOVS'
    AND LEFT(security_isin, 2) IN ('DE')
    AND assttp_scty_issr_sector_riad = 'S1311'
GROUP BY s.business_date
ORDER BY s.business_date

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    SUM(s.nominal_value) AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.nominal_ccy IN ('EUR', 'USD')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND ((s_lender.sector = 'BANK' AND lender_country = 'GB') OR lender_country_residence = 'KY')
    AND security_type = 'GOVS'
    AND LEFT(security_isin, 2) IN ('DE')
    AND assttp_scty_issr_sector_riad = 'S1311'
GROUP BY s.business_date
ORDER BY s.business_date
"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df = df_borrowing.merge(df_lending, on= ['business_date'], how = 'outer')

In [ ]:
df_pivoted['business_date'] = pd.to_datetime(df_pivoted['business_date'])

In [ ]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [ ]:
df_full = df.merge(df_pivoted[['business_date', 'vol_sell', 'vol_buy']], on = 'business_date', how = 'left')

In [ ]:
df_full['net_repo'] = (df_full['borrowing_volume'] - df_full['lending_volume'])/1e9

In [ ]:
df_full['net_futures'] = (df_full['vol_sell'] - df_full['vol_buy'])/1e9

In [ ]:
df_full[['business_date', 'net_futures', 'net_repo']].to_csv('Data\\DE_netpositions_limitations.csv', index = False)

### only for italy

In [ ]:
index_names = tuple(['EURO-BTP LONGTERM ITA GOV BOND', 'Euro-BTP Italian Government Bond',
    'Euro BTP Futures', '"EUX EURO-BTP"', 'Long-Term Euro-BTP',
    'BTP FUTURES JUN24', 'BTP FUTURES SEP24', 'BTP FUTURES DEC24',
    'BTP FUTURES MAR25', 'BTP FUTURES JUN25', 'BTP FUTURES SEP25',
    'Euro-BTP-Italian-Government-Bond', 'Euro BTP Futures',
    'SHORT TERM EURO-BTP  bond', 'Short-Term Euro-BTP Italian Government Bond',
    'SHORT TERM BTP FUTURES', 'Short-Term Euro-BTP',
    'Short-Term-Euro-BTP-Italian-Government-Bond'])

In [ ]:
product_isin = tuple([
    'DE000A0ZW3V8','DE000A1EZJ09',

    'DE000F1B2NF9','DE000F1NGF38','DE000F1ZLHN3',
    'DE000F2MGDA5','DE000F2Y2EW5','DE000F3A9ER7',

    'DE000F1B2NE2','DE000F1NGF46','DE000F1ZLHM5',
    'DE000F2MGDB3','DE000F2Y2EX3','DE000F3A9ES5',

    'DE000F0Q7BS0',

    'DE000C6YTCK8','DE000C5GTX90','DE000C5RQD86','DE000C52GUC3',
    'DE000C58EEE5','DE000C58EEF2',
    'DE000C6EBR57','DE000C6EBR65',
    'DE000C1T6JD0','DE000C6J0JZ3','DE000C6R0411','DE000C671E06',

    'DE000C75XMT0','DE000F0FSQM5','DE000F0Q7BT8','DE000F01NAC1',
    'DE000C5GTXA4','DE000C5RQD94','DE000C52GUD1','DE000C1T6JC2',
    'DE000C6J0K03','DE000C6R0437','DE000C6YTCL6','DE000C671E14',
    'DE000C7F2X93','DE000C7PB9X7',

    'DE000C75XMS2','DE000F0FSQL7','DE000F01NAB3',
    'DE000C7F2X85','DE000C7PB9W9','DE000C7X7UH4','DE000C7X7UJ0'
])

In [ ]:
# Data prep
query = f"""

SELECT reference_period as business_date, notional_leg1_currency as currency, CASE WHEN direction = 'MAKE' THEN 'SLLR' WHEN direction = 'TAKE' THEN 'BYER' ELSE direction END AS direction, SUM(notional_leg1) as vol
FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
WHERE reference_period >= '2024-04-29'
AND reference_period <= '2025-12-31'
AND notional_leg1 < 1e18
AND asset_class = 'INTR'
AND contract_type = 'FUTR'
AND notional_leg1 > 0
AND notional_leg1_currency IN ("EUR")
AND LEFT(product_cfi, 3) = 'FFD'
AND (LEFT(underlying_id, 2) != 'EU' OR underlying_id IS NULL)
AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
AND MONTH(maturity_date) IN (3, 6, 9, 12)
AND (((s_entity.sector = 'BANK' OR s_entity.sector = 'OFI') AND other_cpty_id_country_code_gleif = 'GB') OR other_cpty_id_country_code_gleif = 'KY')
AND ccp_id = '529900LN3S50JPU47S06'
AND notional_leg1 <= 5e10
AND (underlying_id IN {product_isin} OR product_isin IN {product_isin} OR underlying_index_name IN {index_names})
GROUP BY reference_period, currency, direction
ORDER BY reference_period, currency, direction

"""

df_emir_refit = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""


SELECT reference_period as business_date, notional_currency1 as currency,  counterparty_side as direction, SUM(notional) as vol
FROM crp_emir_ecb.emir_ecb_states_validated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_entity ON s.other_cpty_id = s_entity.id
WHERE reference_period >= '2021-01-04'
AND reference_period <= '2024-04-29'
AND notional < 1e18
AND asset_class = 'INTR'
AND contract_type = 'FUTR'
AND notional > 0
AND notional_currency1 IN ("EUR")
AND LEFT(product_clssfctn, 3) = 'FFD'
AND ((LEFT(underlying_id, 2) NOT IN ('EU', 'XC')) OR underlying_id IS NULL)
AND (underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR underlying_index_name IS NULL)
AND MONTH(maturity_date) IN (3, 6, 9, 12)
AND (((s_entity.sector = 'BANK' OR s_entity.sector = 'OFI') AND other_cpty_id_country_code = 'GB') OR other_cpty_id_country_code = 'KY')
AND ccp_id = '529900LN3S50JPU47S06'
AND notional <= 5e10
AND (underlying_id IN {product_isin} OR product_id IN {product_isin} OR underlying_index_name IN {index_names})
GROUP BY reference_period, currency, direction
ORDER BY reference_period, currency, direction


"""

df_emir_old = pd.read_sql_query(query, cnxn)

In [ ]:
df_emir = pd.concat([df_emir_refit, df_emir_old]).drop_duplicates().reset_index(drop=True)

In [ ]:
df_pivoted = df_emir.pivot_table(
    index=['business_date', 'currency'],
    columns='direction',
    values='vol',
    aggfunc='sum',
    fill_value=0
).reset_index()

df_pivoted.columns.name = None  # remove the 'direction' label on columns
df_pivoted = df_pivoted.rename(columns={'SLLR': 'vol_sell', 'BYER': 'vol_buy'})

In [ ]:
# data prep
query = f"""

SELECT 
    s.business_date,
    SUM(s.nominal_value) AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND ((s_borrower.sector = 'BANK' AND borrower_country = 'GB') OR borrower_country_residence = 'KY')
    AND security_type = 'GOVS'
    AND LEFT(security_isin, 2) IN ('IT')
    AND assttp_scty_issr_sector_riad = 'S1311'
GROUP BY s.business_date
ORDER BY s.business_date

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    SUM(s.nominal_value) AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.nominal_ccy IN ('EUR', 'USD')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND ((s_lender.sector = 'BANK' AND lender_country = 'GB') OR lender_country_residence = 'KY')
    AND security_type = 'GOVS'
    AND LEFT(security_isin, 2) IN ('IT')
    AND assttp_scty_issr_sector_riad = 'S1311'
GROUP BY s.business_date
ORDER BY s.business_date
"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df = df_borrowing.merge(df_lending, on= ['business_date'], how = 'outer')

In [ ]:
df_pivoted['business_date'] = pd.to_datetime(df_pivoted['business_date'])

In [ ]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [ ]:
df_full = df.merge(df_pivoted[['business_date', 'vol_sell', 'vol_buy']], on = 'business_date', how = 'left')

In [ ]:
df_full['net_repo'] = (df_full['borrowing_volume'] - df_full['lending_volume'])/1e9

In [ ]:
df_full['net_futures'] = (df_full['vol_sell'] - df_full['vol_buy'])/1e9

In [ ]:
df_full[['business_date', 'net_futures', 'net_repo']].to_csv('Data\\IT_netpositions_limitations.csv', index = False)

# Sector breakdown of German futures, sector_breakdown_german_futures.csv

In [ ]:
german_futures = pd.read_csv('Data\\german_futures.csv', index_col = 0)

In [ ]:
futures = tuple(german_futures['0'][german_futures['0'].isna() == False].unique())

In [ ]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [ ]:
hfunds = tuple(hedge_funds['entity_id'].unique())

In [ ]:
# Data prep
query = f"""

SELECT reference_period as business_date, CASE WHEN leg1_direction = 'MAKE' THEN 'SLLR' WHEN leg1_direction = 'TAKE' THEN 'BYER' ELSE leg1_direction END AS direction, SUM(leg1_notional_leg1) as vol, 
CASE WHEN leg1_other_cpty_id_country_code_gleif IN ('FR', 'DE', 'LU', 'CY', 'IE', 'NL', 'ES', 'IT', 'PT', 'AT', 'FI', 'BE', 'SI', 'GR', 'EE', 'LT', 'HR', 'MT', 'SK', 'LV') THEN 'EA' ELSE 'non-EA' END AS other_country,
CASE WHEN leg1_reporting_cpty_id_country_code_gleif IN ('FR', 'DE', 'LU', 'CY', 'IE', 'NL', 'ES', 'IT', 'PT', 'AT', 'FI', 'BE', 'SI', 'GR', 'EE', 'LT', 'HR', 'MT', 'SK', 'LV') THEN 'EA' ELSE 'non-EA' END AS reporter_country,
CASE 
    WHEN s_reporter.sector = 'BANK' THEN 'BANK' 
    WHEN s_reporter.sector = 'IC' OR s_reporter.sector = 'PF' THEN 'ICPF'
    WHEN leg1_reporting_cpty_id IN {hfunds} OR s_reporter.if_strategy = 'Hedge' THEN 'HF'
    WHEN s_reporter.subsector = 'UCIT' THEN 'UCIT'
    WHEN s_reporter.sector = 'IF' THEN 'IF'
    WHEN s_reporter.sector = 'GOVT' THEN 'GOVT'
    WHEN s_reporter.sector = 'OFI' THEN 'OFI'
    ELSE 'Other'
END as reporter_sector,
CASE 
    WHEN s_other.sector = 'BANK' THEN 'BANK' 
    WHEN s_other.sector = 'IC' OR s_other.sector = 'PF' THEN 'ICPF'
    WHEN leg1_other_cpty_id IN {hfunds} OR s_other.if_strategy = 'Hedge' THEN 'HF'
    WHEN s_other.sector = 'IF' THEN 'IF'
    WHEN s_other.sector = 'GOVT' THEN 'GOVT'
    WHEN s_other.sector = 'OFI' THEN 'OFI'
    ELSE 'Other'
END as other_sector
FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_reporter ON s.leg1_reporting_cpty_id = s_reporter.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_other ON s.leg1_other_cpty_id = s_other.id
WHERE reference_period >= '2024-04-29'
AND reference_period <= '2025-12-31'
AND leg1_asset_class = 'INTR'
AND leg1_contract_type = 'FUTR'
AND leg1_notional_leg1 > 0
AND leg1_notional_leg1_currency IN ("EUR")
AND LEFT(leg1_product_cfi, 3) = 'FFD'
AND (LEFT(leg1_underlying_id, 2) != 'EU' OR leg1_underlying_id IS NULL)
AND (leg1_underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR leg1_underlying_index_name IS NULL)
AND MONTH(leg1_maturity_date) IN (3, 6, 9, 12)
AND leg1_product_isin IN {futures}
AND leg1_notional_leg1 <= 1e11
GROUP BY reference_period, reporter_country, other_country, reporter_sector, other_sector, direction
ORDER BY reference_period, reporter_country, other_country, reporter_sector, other_sector, direction

"""

df_emir_refit = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""


SELECT reference_period as business_date, leg1_counterparty_side as direction, SUM(leg1_notional) as vol, 
CASE WHEN leg1_other_cpty_id_country_code IN ('FR', 'DE', 'LU', 'CY', 'IE', 'NL', 'ES', 'IT', 'PT', 'AT', 'FI', 'BE', 'SI', 'GR', 'EE', 'LT', 'HR', 'MT', 'SK', 'LV') THEN 'EA' ELSE 'non-EA' END AS other_country,
CASE WHEN leg1_reporting_cpty_id_country_code IN ('FR', 'DE', 'LU', 'CY', 'IE', 'NL', 'ES', 'IT', 'PT', 'AT', 'FI', 'BE', 'SI', 'GR', 'EE', 'LT', 'HR', 'MT', 'SK', 'LV') THEN 'EA' ELSE 'non-EA' END AS reporter_country,
CASE 
    WHEN s_reporter.sector = 'BANK' THEN 'BANK' 
    WHEN s_reporter.sector = 'IC' OR s_reporter.sector = 'PF' THEN 'ICPF'
    WHEN leg1_reporting_cpty_id IN {hfunds} OR s_reporter.if_strategy = 'Hedge' THEN 'HF'
    WHEN s_reporter.subsector = 'UCIT' THEN 'UCIT'
    WHEN s_reporter.sector = 'IF' THEN 'IF'
    WHEN s_reporter.sector = 'GOVT' THEN 'GOVT'
    WHEN s_reporter.sector = 'OFI' THEN 'OFI'
    ELSE 'Other'
END as reporter_sector,
CASE 
    WHEN s_other.sector = 'BANK' THEN 'BANK' 
    WHEN s_other.sector = 'IC' OR s_other.sector = 'PF' THEN 'ICPF'
    WHEN leg1_other_cpty_id IN {hfunds} OR s_other.if_strategy = 'Hedge' THEN 'HF'
    WHEN s_other.sector = 'IF' THEN 'IF'
    WHEN s_other.sector = 'GOVT' THEN 'GOVT'
    WHEN s_other.sector = 'OFI' THEN 'OFI'
    ELSE 'Other'
END as other_sector
FROM crp_emir_ecb.emir_ecb_states_deduplicated s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_reporter ON s.leg1_reporting_cpty_id = s_reporter.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20250328 s_other ON s.leg1_other_cpty_id = s_other.id
WHERE reference_period >= '2021-01-04'
AND reference_period <= '2024-04-29'
AND leg1_asset_class = 'INTR'
AND leg1_contract_type = 'FUTR'
AND leg1_notional > 0
AND leg1_notional_currency1 IN ("EUR")
AND LEFT(leg1_product_clssfctn, 3) = 'FFD'
AND ((LEFT(leg1_underlying_id, 2) NOT IN ('EU', 'XC')) OR leg1_underlying_id IS NULL)
AND (leg1_underlying_index_name NOT IN ('United States SOFR Secured Overnight Financing Rat', 'FEDERAL FUNDS EFFECTIVE RATE US') OR leg1_underlying_index_name IS NULL)
AND MONTH(leg1_maturity_date) IN (3, 6, 9, 12)
AND leg1_product_id IN {futures}
AND leg1_notional <= 1e11
GROUP BY reference_period, reporter_country, other_country, reporter_sector, other_sector, direction
ORDER BY reference_period, reporter_country, other_country, reporter_sector, other_sector, direction


"""

df_emir_old = pd.read_sql_query(query, cnxn)

In [ ]:
df_emir = pd.concat([df_emir_refit, df_emir_old]).drop_duplicates().reset_index(drop=True)

In [ ]:
df_emir['other_sector'].unique()

In [ ]:
df_emir.columns

In [ ]:
# Build the reporter-side view
df_reporter = df_emir[['business_date', 'reporter_sector', 'reporter_country', 'direction', 'vol']].copy()
df_reporter = df_reporter.rename(columns={
    'reporter_sector': 'sector',
    'reporter_country': 'country'
})
# Reporter's direction is as-reported
df_reporter['direction'] = df_reporter['direction'].map({'BYER': 'long_pos', 'SLLR': 'short_pos'})

# Build the other-side view
df_other = df_emir[['business_date', 'other_sector', 'other_country', 'direction', 'vol']].copy()
df_other = df_other.rename(columns={
    'other_sector': 'sector',
    'other_country': 'country'
})
# Other counterparty's direction is the OPPOSITE of what the reporter reported
df_other['direction'] = df_other['direction'].map({'BYER': 'short_pos', 'SLLR': 'long_pos'})

# Stack
df_emir = pd.concat([df_reporter, df_other], ignore_index=True)

In [ ]:
df_emir['sector'] = df_emir['sector'] + '_' + df_emir['country']

In [ ]:
df_emir.sample(5)

In [ ]:
# Pivot
df_emir = df_emir.pivot_table(
    index=['business_date', 'sector'],
    columns='direction',
    values='vol',
    aggfunc='sum',
    fill_value=0
).reset_index()

df_emir.columns.name = None

In [ ]:
df_emir['net'] = (df_emir['long_pos'] - df_emir['short_pos'])/1e9

In [ ]:
df_emir.to_csv('Data\\sector_breakdown_german_futures.csv', index= False)

# Fund dealer panels, fund_dealer_day csv files, dealer_cds.csv, dealer_book_day.csv, fund_dealer_bond_day.csv, bond_day_rate.csv, fund_dealer_country_day.csv

In [ ]:
isin = pd.read_excel('Data\\EA_ISINs.xlsx')

In [ ]:
unique_isin = tuple(isin['ISIN'])

In [ ]:
isin['ISIN'].str[:2].unique()

In [ ]:
treasury = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [ ]:
unique_treasury = tuple(treasury['ISIN'].unique())

In [ ]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [ ]:
hf_overlap = tuple(hedge_funds['entity_id'].unique())

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS borrowing_volume, 
    AVG(haircut)                AS borrowing_haircut, 
    AVG(CASE WHEN contractual_maturity <= 0 THEN 1 ELSE contractual_maturity END) AS borrowing_tenor
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id
ORDER BY s.business_date, s.borrower_id, s.lender_id

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS lending_volume, 
    AVG(haircut)                AS lending_haircut, 
    AVG(CASE WHEN contractual_maturity <= 0 THEN 1 ELSE contractual_maturity END) AS lending_tenor
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id
ORDER BY s.business_date, s.lender_id, s.borrower_id
"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df = df_borrowing.merge(df_lending, on= ['business_date', 'fund_id', 'dealer_id'], how = 'outer')

In [ ]:
df.to_csv('key dataframe\\fund_dealer_day.csv')

## USD version of the fund x dealer x day panel

The same two queries as above with USD repos on Treasuries, for the US dealer concentration figure in the slides

In [ ]:
# Data prep
# USD repos collateralised by Treasuries, otherwise identical to the EUR query above
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS borrowing_volume, 
    AVG(haircut)                AS borrowing_haircut, 
    AVG(CASE WHEN contractual_maturity <= 0 THEN 1 ELSE contractual_maturity END) AS borrowing_tenor
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('USD')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_treasury}
GROUP BY s.business_date, s.borrower_id, s.lender_id
ORDER BY s.business_date, s.borrower_id, s.lender_id

"""

df_borrowing_usd = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS lending_volume, 
    AVG(haircut)                AS lending_haircut, 
    AVG(CASE WHEN contractual_maturity <= 0 THEN 1 ELSE contractual_maturity END) AS lending_tenor
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('USD')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_treasury}
GROUP BY s.business_date, s.lender_id, s.borrower_id
ORDER BY s.business_date, s.lender_id, s.borrower_id
"""

df_lending_usd = pd.read_sql_query(query, cnxn)

In [ ]:
df_usd = df_borrowing_usd.merge(df_lending_usd, on= ['business_date', 'fund_id', 'dealer_id'], how = 'outer')

In [ ]:
df_usd.to_csv('key dataframe\\fund_dealer_day_USD.csv')

In [ ]:
cds_map = pd.read_excel('Data\\dealer_bloomberg.xlsx', sheet_name='Sheet1')
cds_wide = pd.read_excel('Data\\dealer_bloomberg.xlsx', sheet_name='Sheet3')

In [ ]:
cds_wide = cds_wide.loc[:, ~cds_wide.columns.str.endswith('.1')] # tickers shared by two LEIs were pulled twice
cds_long = cds_wide.melt(id_vars='Dates', var_name='Bloomberg', value_name='cds')
cds_long = cds_long.merge(cds_map[['dealer_id', 'Bloomberg', 'nationality']], on='Bloomberg', how='inner') # every LEI gets its parent's series
cds_long['period'] = cds_long['Dates'].dt.strftime('%Y-%m-%d')
cds_long = cds_long[['dealer_id', 'Bloomberg', 'nationality', 'period', 'cds']].rename(columns={'Bloomberg': 'bloomberg'})

In [ ]:
cds_long

In [ ]:
cds_long.to_csv('key dataframe\\dealer_cds.csv')

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND (s_lender.sector <> 'HF' OR s_lender.sector IS NULL)
GROUP BY s.business_date, s.borrower_id
ORDER BY s.business_date, s.borrower_id
"""

df_book_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND (s_borrower.sector <> 'HF' OR s_borrower.sector IS NULL)
GROUP BY s.business_date, s.lender_id
ORDER BY s.business_date, s.lender_id
"""

df_book_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_book = df_book_borrowing.merge(df_book_lending, on= ['business_date', 'dealer_id'], how = 'outer')

In [ ]:
df_book.to_csv('key dataframe\\dealer_book_day.csv')

In [ ]:
# Data prep
# rates by fund x dealer x bond x day, simple averages over the trades that carry a rate,
# the trade count lets Stata rebuild the rate sum and leave the fund out of the bond day benchmark
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    s.security_isin,
    SUM(s.nominal_value)                                                                AS borrowing_volume, 
    AVG(s.repo_rate)                                                                    AS borrowing_rate, 
    COUNT(s.repo_rate)                                                                  AS borrowing_trades
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id, s.security_isin
ORDER BY s.business_date, s.borrower_id, s.lender_id, s.security_isin

"""

df_rate_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    s.security_isin,
    SUM(s.nominal_value)                                                                AS lending_volume, 
    AVG(s.repo_rate)                                                                    AS lending_rate, 
    COUNT(s.repo_rate)                                                                  AS lending_trades
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id, s.security_isin
ORDER BY s.business_date, s.lender_id, s.borrower_id, s.security_isin
"""

df_rate_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_rate = df_rate_borrowing.merge(df_rate_lending, on= ['business_date', 'fund_id', 'dealer_id', 'security_isin'], how = 'outer')

In [ ]:
df_rate.to_csv('key dataframe\\fund_dealer_bond_day.csv')

In [ ]:
# Data prep
# the bond day benchmark over everything, all sectors, same filters otherwise,
# the counts give the cell size for a minimum number of counterparties
query = f"""

SELECT 
    s.business_date,
    s.security_isin,
    AVG(s.repo_rate)                                                                    AS market_rate, 
    COUNT(s.repo_rate)                                                                  AS market_trades, 
    COUNT(DISTINCT s.borrower_id)                                                       AS market_borrowers, 
    COUNT(DISTINCT s.lender_id)                                                         AS market_lenders
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.security_isin
ORDER BY s.business_date, s.security_isin
"""

df_market = pd.read_sql_query(query, cnxn)

In [ ]:
df_market.to_csv('key dataframe\\bond_day_rate.csv')

## Dealer fragility by collateral country

The same two queries as for the fund x dealer x day panel, with the collateral country added as the first two letters of the ISIN. The dealer book is unchanged, the treatment stays at the dealer level.

In [ ]:
# Data prep
# fund x dealer x collateral country x day, the country is the first two letters of the ISIN
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    LEFT(s.security_isin, 2) AS country,
    SUM(s.nominal_value)                                                                AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id, LEFT(s.security_isin, 2)
ORDER BY s.business_date, s.borrower_id, s.lender_id, LEFT(s.security_isin, 2)

"""

df_country_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    LEFT(s.security_isin, 2) AS country,
    SUM(s.nominal_value)                                                                AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id, LEFT(s.security_isin, 2)
ORDER BY s.business_date, s.lender_id, s.borrower_id, LEFT(s.security_isin, 2)

"""

df_country_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_country = df_country_borrowing.merge(df_country_lending, on= ['business_date', 'fund_id', 'dealer_id', 'country'], how = 'outer')

In [ ]:
df_country.to_csv('key dataframe\\fund_dealer_country_day.csv')

# Summary statistics for the EUR and USD repo table in the slides

In [ ]:
isin = pd.read_excel('Data\\EA_ISINs.xlsx')

In [ ]:
unique_isin = tuple(isin['ISIN'])

In [ ]:
treasury = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [ ]:
unique_treasury = tuple(treasury['ISIN'].unique())

## Do Funds trade both EUR and USD?

In [ ]:
# Data prep
query = f"""

SELECT DISTINCT borrower_id as entity_id
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}

"""

df_borrowing_eur = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT DISTINCT lender_id as entity_id
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}

"""

df_lending_eur = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT DISTINCT borrower_id as entity_id
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}

"""

df_borrowing_usd = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT DISTINCT lender_id as entity_id
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}

"""

df_lending_usd = pd.read_sql_query(query, cnxn)

In [ ]:
# After pulling EUR and USD data
eur_entities = set(df_borrowing_eur['entity_id'].unique()) | set(df_lending_eur['entity_id'].unique())
usd_entities = set(df_borrowing_usd['entity_id'].unique()) | set(df_lending_usd['entity_id'].unique())

both = eur_entities & usd_entities
eur_only = eur_entities - usd_entities
usd_only = usd_entities - eur_entities

print(f"EUR only: {len(eur_only)}")
print(f"USD only: {len(usd_only)}")
print(f"Both:     {len(both)}")
print(f"Total:    {len(eur_entities | usd_entities)}")

## how about volumes

In [ ]:
# Data prep
query = f"""

SELECT borrower_id as entity_id, SUM(nominal_value) as volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}
GROUP BY entity_id

"""

df_borrowing_eur = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT lender_id as entity_id, SUM(nominal_value) as volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}
GROUP BY entity_id

"""

df_lending_eur = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT borrower_id as entity_id, SUM(nominal_value) as volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}
GROUP BY entity_id

"""

df_borrowing_usd = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT lender_id as entity_id, SUM(nominal_value) as volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}
GROUP BY entity_id

"""

df_lending_usd = pd.read_sql_query(query, cnxn)

In [ ]:
# Combine borrowing + lending volumes per entity per currency
eur_entities = set(df_borrowing_eur['entity_id'].unique()) | set(df_lending_eur['entity_id'].unique())
usd_entities = set(df_borrowing_usd['entity_id'].unique()) | set(df_lending_usd['entity_id'].unique())

both = eur_entities & usd_entities

eur_vol = pd.concat([df_borrowing_eur, df_lending_eur]).groupby('entity_id')['volume'].sum().reset_index()
usd_vol = pd.concat([df_borrowing_usd, df_lending_usd]).groupby('entity_id')['volume'].sum().reset_index()

# Tag each entity
eur_vol['category'] = eur_vol['entity_id'].apply(lambda x: 'Both' if x in both else 'EUR only')
usd_vol['category'] = usd_vol['entity_id'].apply(lambda x: 'Both' if x in both else 'USD only')

# Volume shares
eur_shares = eur_vol.groupby('category')['volume'].sum()
print("EUR volume shares:")
print(eur_shares / eur_shares.sum())

usd_shares = usd_vol.groupby('category')['volume'].sum()
print("\nUSD volume shares:")
print(usd_shares / usd_shares.sum())

In [ ]:
eur_vol_all = eur_vol.sort_values('volume', ascending=False)
eur_vol_all['cumshare'] = eur_vol_all['volume'].cumsum() / eur_vol_all['volume'].sum()
print(eur_vol_all[['entity_id', 'cumshare']].head(20))

In [ ]:
usd_vol_all = usd_vol.sort_values('volume', ascending=False)
usd_vol_all['cumshare'] = usd_vol_all['volume'].cumsum() / usd_vol_all['volume'].sum()
print(usd_vol_all[['entity_id', 'cumshare']].head(20))

## Counterparties of Hedge funds

In [ ]:
# Data prep
query = f"""

SELECT DISTINCT lender_id as entity_id
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}

"""

df_cp_eur_1 = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT DISTINCT borrower_id as entity_id
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}

"""

df_cp_eur_2 = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT DISTINCT lender_id as entity_id
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}

"""

df_cp_usd_1 = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT DISTINCT borrower_id as entity_id
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}

"""

df_cp_usd_2 = pd.read_sql_query(query, cnxn)

In [ ]:
eur_cp = set(df_cp_eur_1['entity_id']) | set(df_cp_eur_2['entity_id'])
usd_cp = set(df_cp_usd_1['entity_id']) | set(df_cp_usd_2['entity_id'])

both_cp = eur_cp & usd_cp
eur_only_cp = eur_cp - usd_cp
usd_only_cp = usd_cp - eur_cp

print(f"EUR only: {len(eur_only_cp)}")
print(f"USD only: {len(usd_only_cp)}")
print(f"Both:     {len(both_cp)}")
print(f"Total:    {len(eur_cp | usd_cp)}")

## How about volumes

In [ ]:
# Data prep
query = f"""

SELECT lender_id as entity_id, SUM(nominal_value) as volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}
GROUP BY entity_id

"""

df_cp_eur_vol_1 = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT borrower_id as entity_id, SUM(nominal_value) as volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}
GROUP BY entity_id

"""

df_cp_eur_vol_2 = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT lender_id as entity_id, SUM(nominal_value) as volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}
GROUP BY entity_id

"""

df_cp_usd_vol_1 = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT borrower_id as entity_id, SUM(nominal_value) as volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}
GROUP BY entity_id

"""

df_cp_usd_vol_2 = pd.read_sql_query(query, cnxn)

In [ ]:
eur_cp = set(df_cp_eur_1['entity_id']) | set(df_cp_eur_2['entity_id'])
usd_cp = set(df_cp_usd_1['entity_id']) | set(df_cp_usd_2['entity_id'])

both_cp = eur_cp & usd_cp
eur_only_cp = eur_cp - usd_cp
usd_only_cp = usd_cp - eur_cp

In [ ]:
eur_cp_vol = pd.concat([df_cp_eur_vol_1, df_cp_eur_vol_2]).groupby('entity_id')['volume'].sum().reset_index()
usd_cp_vol = pd.concat([df_cp_usd_vol_1, df_cp_usd_vol_2]).groupby('entity_id')['volume'].sum().reset_index()

eur_cp_vol['category'] = eur_cp_vol['entity_id'].apply(lambda x: 'Both' if x in both_cp else 'EUR only')
usd_cp_vol['category'] = usd_cp_vol['entity_id'].apply(lambda x: 'Both' if x in both_cp else 'USD only')

eur_shares = eur_cp_vol.groupby('category')['volume'].sum()
print("EUR counterparty volume shares:")
print(eur_shares / eur_shares.sum())

usd_shares = usd_cp_vol.groupby('category')['volume'].sum()
print("\nUSD counterparty volume shares:")
print(usd_shares / usd_shares.sum())

In [ ]:
eur_cp_both = eur_cp_vol.sort_values('volume', ascending=False)
eur_cp_both['cumshare'] = eur_cp_both['volume'].cumsum() / eur_cp_both['volume'].sum()
print(eur_cp_both[['entity_id', 'cumshare']].head(10))

# these are primary dealers, US banks, and EA banks

In [ ]:
# primary dealer share
0.240456 + (0.615474 - 0.436240) + (0.897876 - 0.853970)

In [ ]:
usd_cp_both = usd_cp_vol.sort_values('volume', ascending=False)
usd_cp_both['cumshare'] = usd_cp_both['volume'].cumsum() / usd_cp_both['volume'].sum()
print(usd_cp_both[['entity_id', 'cumshare']].head(10))

# the first three are primary dealers, so they account for 88% of activity in USD

## are hedge funds always transacting with the same arm of the banking group?

In [ ]:
# Data prep
query = f"""

SELECT
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END AS a,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END AS b
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
WHERE intragroup = 1
GROUP BY
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END;


"""

df_edges = pd.read_sql_query(query, cnxn)

In [ ]:
import pandas as pd

# edges: columns ['a','b']
edges = df_edges.values.tolist()

parent = {}
rank = {}

def find(x):
    if parent.setdefault(x, x) != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    rx, ry = find(x), find(y)
    if rx == ry: return
    rxr, ryr = rank.setdefault(rx,0), rank.setdefault(ry,0)
    if rxr < ryr:
        parent[rx] = ry
    elif rxr > ryr:
        parent[ry] = rx
    else:
        parent[ry] = rx
        rank[rx] = rxr + 1

for a,b in edges:
    union(a,b)

# canonical id = min-LEI per component (stable label)
from collections import defaultdict
comps = defaultdict(list)
for x in parent.keys():
    comps[find(x)].append(x)

canon = {rep: min(members) for rep, members in comps.items()}
mapping = []
for x in parent.keys():
    rep = find(x)
    mapping.append((x, canon[rep]))

df_mapping_cc = pd.DataFrame(mapping, columns=["lei","group_id_synth"]).sort_values(["group_id_synth","lei"])

In [ ]:
foreign_bg = ('0W1U67PTV5WY3WYWKD79',
'2138008P9NOMBRMROI73',
'213800A9GT65GAES2V60',
'213800G8QEXN34A2YG53',
'213800GVD8L87R18CT98',
'213800IBT39XQ9C4CP71',
'213800NW35DTWHTMX505',
'213800RZ3GCUXMBGYN59',
'2IGI19DL77OX0HC3ZE78',
'4PQUHN3JPFGFNF3BB653',
'4ZHCHI4KYZG2WVRT8631',
'5299007QVIQ7IO64NX37',
'5493000IQQ05Y25L0V92',
'5493000YPN33HF74SN02',
'54930010P7BUGOECPI58',
'5493002XYZZ0CGQ6CB58',
'54930040QPHHWGT1J432',
'54930050SE0SM7CM2G07',
'5493006PWI2H6PX25403',
'5493007VSMFZCPV1NB83',
'5493008GNQHVI377MY19',
'5493009NLZXZGJDOPC94',
'549300BKWHXYEXPV0328',
'549300FH0WJAPEHTIQ77',
'549300HU9EWFS3CNX640',
'549300KP56LL8NKKFL47',
'549300RSY622D5TQWS42',
'549300SXSTGQY3EA1B18',
'549300WDT1HWUMTUW770',
'571474TGEMMWANRLN572',
'724500AAT1DK36059L16',
'9676007O0UF5YB3QPR03',
'BWS7DNS2Z4NPKPNYKL75',
'HV5W8PGLJ127N2SFSM23'
)

In [ ]:
euro_area_bg = ('0IKLU6X1B10WK7X42C15',
'1VUV7VQFKUOQSJ21A208',
'2138001YBO7CJNQOEE37',
'21380027LW8AF6I1WA03',
'2138003Z5ZVN16GFYV70',
'2138004VZX8CSGPTDX68',
'21380073P7J4PAD91E29',
'213800AGKVL18YKQSV51',
'213800BWHAS44J2C1B28',
'213800D4LHBCXXEEE235',
'213800DBQIB6VBNU5C64',
'213800FKGFR7Q2ACLS83',
'213800G63T4ER4MSVR22',
'213800HV6TP2I5A6MW58',
'213800I92TAU7I3FP232',
'213800KGF4EFNUQKAT69',
'213800OOQOSULB37T658',
'213800ZIGVOZ992FNQ85',
'222100D7H9VRJEH7DU25',
'222100M2PU043YB7YQ06',
'23B6332KMR0JIZLJG565',
'2534006G7F7F1TFC9T77',
'2534006HF1L4YF10UD91',
'253400N2R0RF14JQ0060',
'2549002MVYWWDX54IB83',
'259400QHDOZWMJ103294',
'259400YLRTOBISHBVX41',
'2W8N8UU78PMDQKZENC08',
'31570010000000036567',
'315700GBLUBZ50S45F53',
'315700GXRKZ452JF2U13',
'3M5E1GQGKL17HI6CPN30',
'3U8WV1YX2VMUHH7Z1Q21',
'48510000156ESYOBV122',
'52965FONQ5NZKP0WZL45',
'52990002O5KK6XOGJ020',
'52990010C4NK412ZL440',
'5299004TE2DYMKEAM814',
'5299005UJX6K7BQKV086',
'52990080NNXXLC14OC65',
'529900AQBND3S6YJLY83',
'529900C214QOT3ZYD838',
'529900C4RSSBWXBSY931',
'529900E1WHT64CB20277',
'529900FWTU88V844B672',
'529900GGYMNGRQTDOO93',
'529900K16YGKC8BES892',
'529900T32UL0CP1FZA06',
'529900UKZBMDBDZIXD62',
'529900VA5CNBWXAONR25',
'549300298FD7AS4PPU70',
'54930056IRBXK0Q1FP96',
'549300685QG7DJS55M76',
'5493007RT80TMKY7TO30',
'549300ABE4K96QOCEH37',
'549300CQ9NLEHMRCU505',
'549300DV870NBWY5W279',
'549300DY78U4CMKNHE48',
'549300DYPOFMXOR7XM56',
'549300FOF121DSRG5867',
'549300FR956J8UJDWQ78',
'549300GOF5A5DHWJLV03',
'549300GRXFI7D6PNEA68',
'549300L7YCATGO57ZE10',
'549300LYFYVPUCG6SY25',
'549300NC3SZTETC10349',
'549300NEBDPH0ZXIF850',
'549300Z6OB1D4ZUBD145',
'63540061DPCBNMCGRY22',
'635400CE9HHFB55PEY43',
'635400GQWXFJDCQXW612',
'635400LNHEPZBRNB5D58',
'635400LRAHYBRUZCIH13',
'724500A11NIP5HCVF984',
'815600154F8F91CF6B05',
'8156002070DA4DCBFF31',
'81560027D07F9BDB8436',
'81560038903FF9FA8F80',
'815600522538355AE429',
'8156007395B20763EB44',
'8156009F40F6523F7022',
'815600A32DA05F693F24',
'8EFE15WY4PBBKG6GZI21',
'95980020140005184148',
'9598009X93GQBNHF1L85',
'959800T0J9ADL4GYSS41',
'9695000O0HNLFNR0FB30',
'969500DCEVPV6UIYK220',
'969500QLO3GN6GUB4P61',
'FI6C7E5PBUB3F9K43B44',
'GP5DT10VX1QRQUKVBK64',
'J4CP7MHCXR8DAQMKIL78',
'NNVPP80YIZGEY2314M97',
'RRAN7P32P0W0YY4XQW79',
'SI5RG2M0WQQLZCXKRM20'
)

In [ ]:
bgroups = tuple(set(foreign_bg + euro_area_bg))

In [ ]:
foreign_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(foreign_bg), 'lei'].unique())
euro_area_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(euro_area_bg), 'lei'].unique())

In [ ]:
def build_mapping_cte(df):
    # Escape single quotes inside LEIs or group IDs safely
    parts = []
    for _, row in df.iterrows():
        lei = str(row["lei"]).replace("'", "''")
        gid = str(row["group_id_synth"]).replace("'", "''")
        parts.append(f"SELECT '{lei}' AS lei, '{gid}' AS group_id_synth")
    return "\nUNION ALL\n".join(parts)

mapping_rows = build_mapping_cte(df_mapping_cc)

In [ ]:
query = f"""

WITH mapping AS (
  {mapping_rows}
)
SELECT s.borrower_id as entity_id, s.lender_id as counterparty_id, 
       m.group_id_synth, SUM(nominal_value) as volume, COUNT(*) as n_trades
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
LEFT JOIN mapping m ON s.lender_id = m.lei
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}
GROUP BY s.borrower_id, s.lender_id, m.group_id_synth
"""

df_eur_1 = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

WITH mapping AS (
  {mapping_rows}
)
SELECT s.lender_id as entity_id, s.borrower_id as counterparty_id, 
       m.group_id_synth, SUM(nominal_value) as volume, COUNT(*) as n_trades
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
LEFT JOIN mapping m ON s.borrower_id = m.lei
WHERE nominal_ccy = 'EUR'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_isin}
GROUP BY s.lender_id, s.borrower_id, m.group_id_synth
"""

df_eur_2 = pd.read_sql_query(query, cnxn)

In [ ]:
df_eur = pd.concat([df_eur_1, df_eur_2])

# For each hedge fund + banking group, count distinct arms
arms_eur = df_eur.groupby(['entity_id', 'group_id_synth']).agg(
    n_arms=('counterparty_id', 'nunique'),
    total_volume=('volume', 'sum'),
    total_trades=('n_trades', 'sum')
).reset_index()

# Split into single-arm vs multi-arm relationships
single = arms_eur[arms_eur['n_arms'] == 1]
multi = arms_eur[arms_eur['n_arms'] > 1]

# Count-based measure
print(f"Single arm: {len(single)} relationships ({len(single)/len(arms_eur):.1%})")
print(f"Multi arm:  {len(multi)} relationships ({len(multi)/len(arms_eur):.1%})")

# Transaction-based measure
print(f"\nSingle arm: {single['total_trades'].sum() / arms_eur['total_trades'].sum():.1%} of trades")
print(f"Multi arm:  {multi['total_trades'].sum() / arms_eur['total_trades'].sum():.1%} of trades")

# Volume-based measure
print(f"\nSingle arm: {single['total_volume'].sum() / arms_eur['total_volume'].sum():.1%} of volume")
print(f"Multi arm:  {multi['total_volume'].sum() / arms_eur['total_volume'].sum():.1%} of volume")

In [ ]:
query = f"""

WITH mapping AS (
  {mapping_rows}
)
SELECT s.borrower_id as entity_id, s.lender_id as counterparty_id, 
       m.group_id_synth, SUM(nominal_value) as volume, COUNT(*) as n_trades
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
LEFT JOIN mapping m ON s.lender_id = m.lei
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (borrower_country_residence = 'KY' AND s_borrower.sector = 'IF' OR s_borrower.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}
GROUP BY s.borrower_id, s.lender_id, m.group_id_synth
"""

df_usd_1 = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

WITH mapping AS (
  {mapping_rows}
)
SELECT s.lender_id as entity_id, s.borrower_id as counterparty_id, 
       m.group_id_synth, SUM(nominal_value) as volume, COUNT(*) as n_trades
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
LEFT JOIN mapping m ON s.borrower_id = m.lei
WHERE nominal_ccy = 'USD'
AND business_date <= '2025-10-01'
AND central_clearing = 'non-cleared'
AND (lender_country_residence = 'KY' AND s_lender.sector = 'IF' OR s_lender.if_strategy = 'Hedge')
AND gnlcoll = 'SPEC'
AND security_isin IN {unique_treasury}
GROUP BY s.lender_id, s.borrower_id, m.group_id_synth
"""

df_usd_2 = pd.read_sql_query(query, cnxn)

In [ ]:
df_usd = pd.concat([df_usd_1, df_usd_2])

# For each hedge fund + banking group, count distinct arms
arms_usd = df_usd.groupby(['entity_id', 'group_id_synth']).agg(
    n_arms=('counterparty_id', 'nunique'),
    total_volume=('volume', 'sum'),
    total_trades=('n_trades', 'sum')
).reset_index()

# Split into single-arm vs multi-arm relationships
single = arms_usd[arms_usd['n_arms'] == 1]
multi = arms_usd[arms_usd['n_arms'] > 1]

# Count-based measure
print(f"Single arm: {len(single)} relationships ({len(single)/len(arms_usd):.1%})")
print(f"Multi arm:  {len(multi)} relationships ({len(multi)/len(arms_usd):.1%})")

# Transaction-based measure
print(f"\nSingle arm: {single['total_trades'].sum() / arms_usd['total_trades'].sum():.1%} of trades")
print(f"Multi arm:  {multi['total_trades'].sum() / arms_usd['total_trades'].sum():.1%} of trades")

# Volume-based measure
print(f"\nSingle arm: {single['total_volume'].sum() / arms_usd['total_volume'].sum():.1%} of volume")
print(f"Multi arm:  {multi['total_volume'].sum() / arms_usd['total_volume'].sum():.1%} of volume")

## do hedge funds face the same banking groups across currencies?

In [ ]:
# Get hedge fund - group pairs per currency
eur_pairs = set(zip(arms_eur['entity_id'], arms_eur['group_id_synth']))
usd_pairs = set(zip(arms_usd['entity_id'], arms_usd['group_id_synth']))

both_pairs = eur_pairs & usd_pairs
eur_only_pairs = eur_pairs - usd_pairs
usd_only_pairs = usd_pairs - eur_pairs

print(f"EUR only: {len(eur_only_pairs)}")
print(f"USD only: {len(usd_only_pairs)}")
print(f"Both:     {len(both_pairs)}")

In [ ]:
# Level 1: volumes by group-pair category
eur_group_vol = df_eur.groupby(['entity_id', 'group_id_synth'])['volume'].sum().reset_index()
usd_group_vol = df_usd.groupby(['entity_id', 'group_id_synth'])['volume'].sum().reset_index()

eur_group_vol['category'] = eur_group_vol.apply(
    lambda r: 'Both' if (r['entity_id'], r['group_id_synth']) in both_pairs else 'EUR only', axis=1)
usd_group_vol['category'] = usd_group_vol.apply(
    lambda r: 'Both' if (r['entity_id'], r['group_id_synth']) in both_pairs else 'USD only', axis=1)

eur_shares = eur_group_vol.groupby('category')['volume'].sum()
print("EUR volume shares:")
print(eur_shares / eur_shares.sum())

usd_shares = usd_group_vol.groupby('category')['volume'].sum()
print("\nUSD volume shares:")
print(usd_shares / usd_shares.sum())

## when they face the same group in both currencies, do they use the same arm?

In [ ]:
# For each hedge fund - group pair that appears in both currencies,
# compare which arms they use
eur_arms = df_eur.groupby(['entity_id', 'group_id_synth'])['counterparty_id'].apply(set).reset_index()
usd_arms = df_usd.groupby(['entity_id', 'group_id_synth'])['counterparty_id'].apply(set).reset_index()

merged = eur_arms.merge(usd_arms, on=['entity_id', 'group_id_synth'], suffixes=('_eur', '_usd'))

merged['same_arms'] = merged.apply(lambda r: r['counterparty_id_eur'] == r['counterparty_id_usd'], axis=1)
merged['any_overlap'] = merged.apply(lambda r: len(r['counterparty_id_eur'] & r['counterparty_id_usd']) > 1, axis=1)
merged['no_overlap'] = merged.apply(lambda r: len(r['counterparty_id_eur'] & r['counterparty_id_usd']) > 0, axis=1)

print(f"Exact same arms:  {merged['same_arms'].sum()} / {len(merged)}")
print(f"At least one shared arm: {merged['any_overlap'].sum()} / {len(merged)}")
print(f"Completely different arms: {(~merged['no_overlap']).sum()} / {len(merged)}")

In [ ]:
eur_vol = df_eur.groupby(['entity_id', 'group_id_synth'])['volume'].sum().reset_index(name='vol_eur')
usd_vol = df_usd.groupby(['entity_id', 'group_id_synth'])['volume'].sum().reset_index(name='vol_usd')

merged = merged.merge(eur_vol, on=['entity_id', 'group_id_synth'])
merged = merged.merge(usd_vol, on=['entity_id', 'group_id_synth'])
merged['total_vol'] = merged['vol_eur'] + merged['vol_usd']

total = merged['total_vol'].sum()
print(f"Exact same arms:  {merged[merged['same_arms']]['total_vol'].sum() / total:.1%}")
print(f"Some overlap:     {merged[merged['any_overlap']]['total_vol'].sum() / total:.1%}")
print(f"No overlap:       {merged[~merged['no_overlap']]['total_vol'].sum() / total:.1%}")

In [ ]:
total_eur = merged['vol_eur'].sum()
total_usd = merged['vol_usd'].sum()

print("EUR:")
print(f"Exact same arms:  {merged[merged['same_arms']]['vol_eur'].sum() / total_eur:.1%}")
print(f"Some overlap:     {merged[merged['any_overlap']]['vol_eur'].sum() / total_eur:.1%}")
print(f"No overlap:       {merged[~merged['no_overlap']]['vol_eur'].sum() / total_eur:.1%}")

print("\nUSD:")
print(f"Exact same arms:  {merged[merged['same_arms']]['vol_usd'].sum() / total_usd:.1%}")
print(f"Some overlap:     {merged[merged['any_overlap']]['vol_usd'].sum() / total_usd:.1%}")
print(f"No overlap:       {merged[~merged['no_overlap']]['vol_usd'].sum() / total_usd:.1%}")

In [ ]:
df_all = pd.concat([df_eur, df_usd])
cp_per_fund = df_all.groupby('entity_id')['counterparty_id'].nunique().reset_index(name='n_counterparties')

print(cp_per_fund['n_counterparties'].describe())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(cp_per_fund['n_counterparties'], bins=range(1, cp_per_fund['n_counterparties'].max() + 2),
        edgecolor='white', color='#2166ac', align='left',
        weights=[1 / len(cp_per_fund)] * len(cp_per_fund))
ax.set_xlabel('Number of counterparties')
ax.set_ylabel('Share of hedge funds')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.tight_layout()
plt.show()

# Bid and ask prices and yields, bond_bidask.csv


In [ ]:
# Bid and ask prices and yields from Bloomberg, Data\bidask_bloomberg.xlsx with one sheet for US and one for EA bonds
# row 1 carries the ISIN above each block of four columns, row 2 the field names, the dates sit in the first column
frames = []
for sheet in ['US', 'EA']:
    raw = pd.read_excel('Data\\bidask_bloomberg.xlsx', sheet_name=sheet, header=None)
    dates = pd.to_datetime(raw.iloc[2:, 0], dayfirst=True, errors='coerce')
    isins = raw.iloc[0].ffill()
    fields = raw.iloc[1]
    for isin in isins.iloc[1:].dropna().unique():
        cols = [c for c in raw.columns[1:] if isins[c] == isin]
        block = raw.iloc[2:, cols].copy()
        block.columns = fields[cols].values
        block = block.rename(columns={'PX_BID': 'bid_price', 'PX_ASK': 'ask_price',
                                      'YLD_YTM_BID': 'bid_yield', 'YLD_YTM_ASK': 'ask_yield'})
        block.insert(0, 'isin', str(isin).replace('isin', '').strip())
        block.insert(0, 'date', dates.values)
        frames.append(block)

df_bidask = pd.concat(frames, ignore_index=True)
for c in ['bid_price', 'ask_price', 'bid_yield', 'ask_yield']:
    df_bidask[c] = pd.to_numeric(df_bidask[c], errors='coerce')  # Bloomberg writes #N/A N/A into cells without data
df_bidask = df_bidask.dropna(subset=['date', 'bid_price', 'ask_price'])  # no price on the day, bonds matured before 2020 drop out entirely
df_bidask = df_bidask.drop_duplicates(subset=['date', 'isin'])
df_bidask = df_bidask.sort_values(['isin', 'date'])
df_bidask = df_bidask[['date', 'isin', 'bid_price', 'ask_price', 'bid_yield', 'ask_yield']]
df_bidask['date'] = df_bidask['date'].dt.strftime('%Y-%m-%d')
df_bidask.to_csv('key dataframe\\bond_bidask.csv', index=False)